In [ ]:
# STEP 1: Read all Data Files from Input/Rental Data

import pandas as pd
import glob
import os

# Define the directory path
input_dir = 'Input/Rental Data'

# Check if directory exists
if not os.path.exists(input_dir):
    print(f"❌ ERROR: Directory '{input_dir}' not found!")
    print(f"Current working directory: {os.getcwd()}")
    print(f"Available directories and files:")
    for item in os.listdir('.'):
        print(f"  - {item}")
else:
    print(f"✅ Directory found: {os.path.abspath(input_dir)}")
    
    # Method 1: Using glob to find all Excel files in Input/Rental Data directory
    excel_files = glob.glob(os.path.join(input_dir, '*.xlsx')) + glob.glob(os.path.join(input_dir, '*.xls'))
    
    # Also look for CSV files if any
    csv_files = glob.glob(os.path.join(input_dir, '*.csv'))
    
    print(f"\n📁 Found {len(excel_files)} Excel file(s) and {len(csv_files)} CSV file(s) in {input_dir} directory:")
    
    # List all Excel files
    if excel_files:
        print("\nExcel files:")
        for i, file in enumerate(excel_files, 1):
            print(f"  {i:2d}. {os.path.basename(file)}")
    
    # List all CSV files
    if csv_files:
        print("\nCSV files:")
        for i, file in enumerate(csv_files, 1):
            print(f"  {i:2d}. {os.path.basename(file)}")
    
    # Create a dictionary to store all DataFrames
    dataframes = {}
    
    # Read each Excel file into a separate DataFrame
    if excel_files:
        print(f"\n{'='*60}")
        print("READING EXCEL FILES")
        print('='*60)
        
        for file in excel_files:
            # Get just the filename without path or extension to use as variable name
            file_name = os.path.splitext(os.path.basename(file))[0]
            
            try:
                # Read the Excel file
                df = pd.read_excel(file)
                
                # Store in dictionary with filename as key
                dataframes[file_name] = df
                
                print(f"✅ Successfully read: {os.path.basename(file)} (shape: {df.shape})")
                
            except Exception as e:
                print(f"❌ Error reading {os.path.basename(file)}: {e}")
    
    # Read each CSV file into a separate DataFrame
    if csv_files:
        print(f"\n{'='*60}")
        print("READING CSV FILES")
        print('='*60)
        
        for file in csv_files:
            # Get just the filename without path or extension to use as variable name
            file_name = os.path.splitext(os.path.basename(file))[0]
            
            try:
                # Read the CSV file
                df = pd.read_csv(file)
                
                # Store in dictionary with filename as key
                dataframes[file_name] = df
                
                print(f"✅ Successfully read: {os.path.basename(file)} (shape: {df.shape})")
                
            except Exception as e:
                print(f"❌ Error reading {os.path.basename(file)}: {e}")
    
    print(f"\n{'='*60}")
    print(f"📊 SUMMARY")
    print(f"{'='*60}")
    print(f"Total files read: {len(dataframes)}")
    
    # Access the dictionary containing all your DataFrames
    if dataframes:
        print("\n📁 Available DataFrames:")
        for i, name in enumerate(sorted(dataframes.keys()), 1):
            print(f"  {i:2d}. {name}")
        
        # Check the structure of the first DataFrame
        sample_key = list(dataframes.keys())[0]
        print(f"\n🔍 Sample DataFrame '{sample_key}':")
        print(dataframes[sample_key].head())
        print(f"\n📋 Columns: {list(dataframes[sample_key].columns)}")
        print(f"📐 Shape: {dataframes[sample_key].shape}")
        
        # Get basic info about all DataFrames
        print(f"\n{'='*60}")
        print("SUMMARY OF ALL DATAFRAMES")
        print(f"{'='*60}")
        
        for name, df in sorted(dataframes.items()):
            print(f"\n📄 {name}:")
            print(f"   Rows: {df.shape[0]:,}, Columns: {df.shape[1]}")
            print(f"   Memory: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
            print(f"   Columns: {list(df.columns)}" if df.shape[1] <= 10 else f"   Columns: {df.shape[1]} columns")
    else:
        print("\n⚠️ No DataFrames were loaded. Check if files exist in the directory.")
        
        # List what's actually in the directory
        print(f"\nContents of '{input_dir}':")
        if os.listdir(input_dir):
            for item in os.listdir(input_dir):
                print(f"  - {item}")
        else:
            print("  (empty directory)")


In [ ]:
# STEP 2: DATA MERGING & CLEANING

# Combine all DataFrames into one
combined_df = pd.concat(dataframes.values(), ignore_index=True)

print(f"Combined DataFrame shape: {combined_df.shape}")
print(f"Total rows: {len(combined_df):,}")
print(f"Total columns: {len(combined_df.columns)}")

# Display the first few rows
print("\nFirst few rows of combined data:")
print(combined_df.head())

# Display basic info about the combined data
print("\nDataFrame info:")
print(combined_df.info())

# Count complete duplicates
complete_duplicates = combined_df.duplicated().sum()
print(f"Complete duplicates (all columns identical): {complete_duplicates}")

# Store original shape for reporting
original_shape = combined_df.shape
print(f"Original data shape: {original_shape}")

# Drop duplicates and keep only the originals
combined_df = combined_df.drop_duplicates()

# Report the results
new_shape = combined_df.shape
rows_removed = original_shape[0] - new_shape[0]

print(f"After removing duplicates: {new_shape}")
print(f"Rows removed: {rows_removed}")
print(f"Unique rows kept: {new_shape[0]}")

# Verify no duplicates remain
remaining_duplicates = combined_df.duplicated().sum()
print(f"Remaining duplicates: {remaining_duplicates}")


In [ ]:
# Basic structure overview
print("=== BASIC STRUCTURE ===")
print(f"DataFrame shape: {combined_df.shape}")
print(f"Number of rows: {combined_df.shape[0]:,}")
print(f"Number of columns: {combined_df.shape[1]}")
print(f"Total cells: {combined_df.size:,}")

# Column information
print("\n=== COLUMN INFORMATION ===")
print("Column names and data types:")
print(combined_df.dtypes)


In [ ]:
# If you are short on time- do not run this code!
# This process takes ~6 hours, skip to next code block to import the already generated file provided you have access to the excel file generated previously

import requests
import pandas as pd
import time

def fast_geocode_singapore_address(street_name: str, project_name: str, postal_district: int) -> tuple:
    """
    Fast geocoding using only Format 1: Project + Street
    """
    if pd.isna(street_name) or str(street_name).strip() == "":
        return None, None, "Missing street name"
    
    # Format 1: Project + Street
    if pd.notna(project_name) and str(project_name).strip() and str(project_name).strip().lower() != 'nan':
        search_query = f"{project_name}, {street_name}"
    else:
        search_query = street_name
    
    base_url = "https://www.onemap.gov.sg/api/common/elastic/search"
    
    params = {
        "searchVal": search_query.strip(),
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=5)
        response.raise_for_status()
        data = response.json()
        
        if data.get("found", 0) > 0:
            first_result = data["results"][0]
            lat = float(first_result.get("LATITUDE"))
            lon = float(first_result.get("LONGITUDE"))
            return lat, lon, "Success"
        else:
            return None, None, "No results"
            
    except Exception:
        return None, None, "Error"

def fast_geocode_dataframe(df, batch_size=200, delay=0.1):
    """
    Fast geocoding with minimal overhead
    """
    df = df.copy()
    df['latitude'] = None
    df['longitude'] = None
    df['geocode_status'] = ''
    
    total_rows = len(df)
    success_count = 0
    
    print(f"🚀 FAST GEOCODING STARTED")
    print(f"Total rows: {total_rows:,}")
    print(f"Batch size: {batch_size}")
    print(f"Delay: {delay}s\n")
    
    start_time = time.time()
    
    for i, (idx, row) in enumerate(df.iterrows()):
        # Geocode quickly
        lat, lon, status = fast_geocode_singapore_address(
            row.get('Street Name', ''), 
            row.get('Project Name', ''), 
            row.get('Postal District', None)
        )
        
        df.at[idx, 'latitude'] = lat
        df.at[idx, 'longitude'] = lon
        df.at[idx, 'geocode_status'] = status
        
        if status == "Success":
            success_count += 1
        
        # Batch progress reporting
        if (i + 1) % batch_size == 0:
            batch_success = success_count
            batch_rate = (batch_success / (i + 1)) * 100
            elapsed = time.time() - start_time
            rows_per_second = (i + 1) / elapsed
            
            print(f"📍 Batch {((i + 1) // batch_size)}: {i + 1:,}/{total_rows:,} rows")
            print(f"   Success: {batch_success:,}/{i + 1:,} ({batch_rate:.1f}%)")
            print(f"   Speed: {rows_per_second:.1f} rows/sec")
            print(f"   Elapsed: {elapsed/60:.1f} min\n")
        
        # Minimal delay
        time.sleep(delay)
    
    # Final results
    total_time = time.time() - start_time
    final_success_rate = (success_count / total_rows) * 100
    
    print("="*50)
    print(f"✅ GEOCODING COMPLETED!")
    print(f"Total time: {total_time/60:.1f} min")
    print(f"Final success rate: {success_count:,}/{total_rows:,} ({final_success_rate:.1f}%)")
    print("="*50)
    
    return df

# RUN IT!
print("Starting fast geocoding...")
# combined_df = fast_geocode_dataframe(combined_df, batch_size=200, delay=0.1)

# Quick save
# combined_df.to_csv("singapore_rental_geocoded_FAST.csv", index=False)
print("💾 Data saved to: singapore_rental_geocoded_FAST.csv")


In [ ]:
# So that you do not need to run the previous cell and wait ~6 hours we are importing the already geocoded file from the previous step

import pandas as pd
import glob
import os

# Read the combined rental data
combined_df = pd.read_csv('Intermediate Files/singapore_rental_geocoded_FAST.csv')

# 1. Drop rows with no geocode from your main rental data
print("=== CLEANING RENTAL DATA ===")
original_rows = len(combined_df)

# Drop rows where geocoding failed
combined_df_clean = combined_df[combined_df['geocode_status'] == 'Success']

rows_removed = original_rows - len(combined_df_clean)
print(f"Original rows: {original_rows:,}")
print(f"Rows with successful geocodes: {len(combined_df_clean):,}")
print(f"Rows removed (no geocode): {rows_removed:,}")
print(f"Success rate: {len(combined_df_clean)/original_rows*100:.1f}%")

# Save cleaned data
combined_df_clean.to_csv("rental_data_geocoded_clean.csv", index=False)
print("💾 Cleaned rental data saved: rental_data_geocoded_clean.csv")

# 2. Load all amenity CSV files from Input/Amenities directory
print("\n" + "="*50)
print("LOADING AMENITY CSV FILES FROM Input/Amenities")
print("="*50)

# Define the amenities directory path
amenities_dir = "Input/Amenities"

# Check if directory exists
if not os.path.exists(amenities_dir):
    print(f"❌ ERROR: Directory '{amenities_dir}' not found!")
    print(f"Current directory: {os.getcwd()}")
    print("Available directories:")
    for item in os.listdir('.'):
        if os.path.isdir(item):
            print(f"  📁 {item}")
    amenity_files = []
else:
    print(f"✅ Directory found: {os.path.abspath(amenities_dir)}")
    
    # Get all CSV files in the amenities directory
    amenity_files = glob.glob(os.path.join(amenities_dir, "*.csv"))
    
    print(f"\nFound {len(amenity_files)} CSV files in {amenities_dir}:")

# Dictionary to store all amenity DataFrames
amenity_dfs = {}

# Load each CSV file and show summary
for file in amenity_files:
    try:
        # Read the CSV file
        df = pd.read_csv(file)
        
        # Get just the filename without path for display
        file_name_short = os.path.basename(file)
        amenity_dfs[file_name_short] = df
        
        print(f"\n📁 {file_name_short}:")
        print(f"   Rows: {len(df):,}")
        print(f"   Columns: {len(df.columns)}")
        print(f"   Columns: {list(df.columns)}")
        
        # Check if has coordinates - look for common column names
        lat_cols = [col for col in df.columns if 'lat' in col.lower()]
        lon_cols = [col for col in df.columns if 'lon' in col.lower() or 'lng' in col.lower()]
        has_coords = len(lat_cols) > 0 and len(lon_cols) > 0
        
        if has_coords:
            print(f"   ✅ Has coordinate columns: {lat_cols[0]}, {lon_cols[0]}")
        else:
            print(f"   ⚠️  No coordinate columns found")
            print(f"      Looking for 'lat'/'lon' or 'latitude'/'longitude'")
            
    except Exception as e:
        print(f"❌ Error loading {file}: {e}")

print(f"\n✅ Successfully loaded {len(amenity_dfs)} amenity files")

# 3. Quick analysis of all loaded data
print("\n" + "="*50)
print("SUMMARY OF ALL DATA")
print("="*50)

total_rows_all = len(combined_df_clean)
print(f"📍 Rental properties: {total_rows_all:,} rows")

for file_name, df in amenity_dfs.items():
    # Remove .csv extension for display
    amenity_name = os.path.splitext(file_name)[0]
    print(f"🏫 {amenity_name}: {len(df):,} rows")

total_amenity_rows = sum(len(df) for df in amenity_dfs.values())
print(f"\n📊 TOTAL AMENITY RECORDS: {total_amenity_rows:,} rows")
print(f"🏠 TOTAL RENTAL PROPERTIES: {total_rows_all:,} rows")
print(f"📈 GRAND TOTAL: {total_rows_all + total_amenity_rows:,} rows")

# 4. Display sample of each amenity type
print("\n" + "="*50)
print("SAMPLE DATA FROM EACH AMENITY TYPE")
print("="*50)

for file_name, df in amenity_dfs.items():
    amenity_name = os.path.splitext(file_name)[0]
    print(f"\n📋 {amenity_name} (first 3 rows):")
    print(df.head(3))
    print("-" * 30)


In [ ]:
import pandas as pd
import numpy as np

combined_df_clean = pd.read_csv('rental_data_geocoded_clean.csv')

# Define amenity files - JUST UPDATED THE PATHS!
amenity_files = {
    'schools': 'Input/Amenities/schools_latlon.csv',
    'parks': 'Input/Amenities/parks_latlon.csv', 
    'mrt_stations': 'Input/Amenities/mrt_station_exits_latlon.csv',
    'gyms': 'Input/Amenities/gyms_latlon.csv',
    'cycling_paths': 'Input/Amenities/cycling_paths_latlon.csv',
    'disability_services': 'Input/Amenities/disability_services_latlon.csv',
    'taxi_stops': 'Input/Amenities/taxi_stops_latlon.csv',
    'tourist_attractions': 'Input/Amenities/tourist_attractions_latlon.csv',
    'hawker_centres': 'Input/Amenities/hawker_centres_latlon.csv',
    'supermarkets': 'Input/Amenities/supermarkets_latlon.csv'
}

print("🔍 CHECKING AMENITY FILES FOR VALID COORDINATES")
print("=" * 70)

for amenity_name, filename in amenity_files.items():
    print(f"\n📁 {amenity_name.upper()} ({filename}):")
    print("-" * 50)
    
    try:
        # Load the file
        df = pd.read_csv(filename)
        total_rows = len(df)
        
        # Check for null values
        null_lat = df['latitude'].isnull().sum()
        null_lon = df['longitude'].isnull().sum()
        
        # Check for zero/invalid coordinates
        zero_lat = (df['latitude'] == 0).sum()
        zero_lon = (df['longitude'] == 0).sum()
        
        # Check for Singapore bounds (roughly lon: 103-104, lat: 1-2)
        valid_sg_coords = df[
            (df['longitude'].between(103.5, 104.5)) &
            (df['latitude'].between(1.0, 2.0))
        ]
        
        # Print results
        print(f"Total rows: {total_rows}")
        print(f"Null latitude: {null_lat} ({null_lat/total_rows*100:.1f}%)")
        print(f"Null longitude: {null_lon} ({null_lon/total_rows*100:.1f}%)")
        print(f"Zero latitude: {zero_lat} ({zero_lat/total_rows*100:.1f}%)")
        print(f"Zero longitude: {zero_lon} ({zero_lon/total_rows*100:.1f}%)")
        print(f"Valid Singapore coordinates: {len(valid_sg_coords)} ({len(valid_sg_coords)/total_rows*100:.1f}%)")
        
        # Show coordinate ranges
        print(f"Latitude range: {df['latitude'].min():.6f} to {df['latitude'].max():.6f}")
        print(f"Longitude range: {df['longitude'].min():.6f} to {df['longitude'].max():.6f}")
        
        # Show first few invalid rows if any
        invalid_coords = df[
            (df['latitude'].isnull()) | 
            (df['longitude'].isnull()) |
            (df['latitude'] == 0) | 
            (df['longitude'] == 0) |
            (~df['longitude'].between(103.5, 104.5)) |
            (~df['latitude'].between(1.0, 2.0))
        ]
        
        if len(invalid_coords) > 0:
            print(f"🚨 Found {len(invalid_coords)} rows with invalid coordinates:")
            print(invalid_coords[['latitude', 'longitude']].head())
        
    except Exception as e:
        print(f"❌ Error reading {filename}: {e}")

print("\n" + "=" * 70)
print("✅ All files checked!")

# Additional: Check your rental data too
print(f"\n📊 RENTAL DATA CHECK (combined_df_clean):")
print("-" * 50)
print(f"Total rows: {len(combined_df_clean)}")
print(f"Null latitude: {combined_df_clean['latitude'].isnull().sum()}")
print(f"Null longitude: {combined_df_clean['longitude'].isnull().sum()}")
print(f"Zero latitude: {(combined_df_clean['latitude'] == 0).sum()}")
print(f"Zero longitude: {(combined_df_clean['longitude'] == 0).sum()}")

# Check Singapore bounds for rental data
valid_rental_coords = combined_df_clean[
    (combined_df_clean['longitude'].between(103.5, 104.5)) &
    (combined_df_clean['latitude'].between(1.0, 2.0))
]
print(f"Valid Singapore coordinates: {len(valid_rental_coords)} ({len(valid_rental_coords)/len(combined_df_clean)*100:.1f}%)")


In [ ]:
!pip install geopy


In [ ]:
# DO NOT RUN THIS CODE IF YOU ARE SHORT ON TIME

import pandas as pd
import numpy as np
from geopy.distance import geodesic
import math
import time

# Load the rental data
print("📁 LOADING RENTAL DATA...")
combined_df_clean = pd.read_csv('rental_data_geocoded_clean.csv')
print(f"✅ Loaded {len(combined_df_clean):,} rental properties")

# Define amenity files
amenity_files = {
    'schools': 'Input/Amenities/schools_latlon.csv',
    'parks': 'Input/Amenities/parks_latlon.csv', 
    'mrt_stations': 'Input/Amenities/mrt_station_exits_latlon.csv',
    'gyms': 'Input/Amenities/gyms_latlon.csv',
    'cycling_paths': 'Input/Amenities/cycling_paths_latlon.csv',
    'disability_services': 'Input/Amenities/disability_services_latlon.csv',
    'taxi_stops': 'Input/Amenities/taxi_stops_latlon.csv',
    'tourist_attractions': 'Input/Amenities/tourist_attractions_latlon.csv',
    'hawker_centres': 'Input/Amenities/hawker_centres_latlon.csv',
    'supermarkets': 'Input/Amenities/supermarkets_latlon.csv'
}

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate great-circle distance between two points using Haversine formula"""
    R = 6371000  # Earth radius in meters
    
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)
    
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = math.sin(dlat/2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    
    return R * c

def calculate_distances_smart(rental_df, amenity_df, amenity_name):
    """SMART approach using grid-based spatial partitioning"""
    
    # Clean amenity data
    amenity_clean = amenity_df.dropna(subset=['latitude', 'longitude']).copy()
    amenity_clean = amenity_clean[
        (amenity_clean['latitude'] != 0) & 
        (amenity_clean['longitude'] != 0)
    ]
    
    print(f"  📍 Using {len(amenity_clean):,} valid {amenity_name}")
    
    # Convert to lists
    rental_coords = list(zip(rental_df['latitude'], rental_df['longitude']))
    amenity_coords = list(zip(amenity_clean['latitude'], amenity_clean['longitude']))
    
    # Create spatial grid for Singapore (divide into smaller search areas)
    grid_size = 0.02  # ~2km grid cells in Singapore
    
    # Singapore bounds
    min_lat, max_lat = 1.2, 1.5
    min_lon, max_lon = 103.6, 104.0
    
    # Assign amenities to grid cells
    grid_amenities = {}
    for amenity_coord in amenity_coords:
        lat, lon = amenity_coord
        grid_key = (int((lat - min_lat) / grid_size), int((lon - min_lon) / grid_size))
        if grid_key not in grid_amenities:
            grid_amenities[grid_key] = []
        grid_amenities[grid_key].append(amenity_coord)
    
    print(f"  🗺️  Created {len(grid_amenities)} spatial grid cells")
    
    distances = []
    total_rentals = len(rental_coords)
    
    for i, rental_coord in enumerate(rental_coords):
        rental_lat, rental_lon = rental_coord
        
        # Find which grid cell the rental is in
        rental_grid = (int((rental_lat - min_lat) / grid_size), int((rental_lon - min_lon) / grid_size))
        
        # Search in this grid cell and adjacent cells
        search_amenities = []
        for lat_offset in [-1, 0, 1]:
            for lon_offset in [-1, 0, 1]:
                search_grid = (rental_grid[0] + lat_offset, rental_grid[1] + lon_offset)
                if search_grid in grid_amenities:
                    search_amenities.extend(grid_amenities[search_grid])
        
        # If no amenities found in nearby grids, search all amenities (fallback)
        if not search_amenities:
            search_amenities = amenity_coords
        
        # Calculate distances only to amenities in nearby grids
        min_dist = min([geodesic(rental_coord, amenity_coord).meters 
                       for amenity_coord in search_amenities])
        distances.append(min_dist)
        
        # Progress update
        if (i + 1) % 5000 == 0:
            print(f"    ✅ Processed {i + 1:,}/{total_rentals:,} properties")
    
    return distances

def calculate_distances_optimized(rental_df, amenity_df, amenity_name):
    """Optimized version for very large amenity datasets"""
    
    # Clean amenity data
    amenity_clean = amenity_df.dropna(subset=['latitude', 'longitude']).copy()
    amenity_clean = amenity_clean[
        (amenity_clean['latitude'] != 0) & 
        (amenity_clean['longitude'] != 0)
    ]
    
    print(f"  📍 Using {len(amenity_clean):,} valid {amenity_name}")
    
    rental_coords = list(zip(rental_df['latitude'], rental_df['longitude']))
    amenity_coords = list(zip(amenity_clean['latitude'], amenity_clean['longitude']))
    
    # For very large datasets, use approximate search with sampling
    if len(amenity_coords) > 2000:
        print("  ⚡ Using sampling optimization for large dataset")
        # Sample amenities to reduce computation (still gives good results for nearest)
        sample_size = min(1000, len(amenity_coords))
        # Use stratified sampling - take from different areas
        amenity_coords_sampled = amenity_coords[::max(1, len(amenity_coords) // sample_size)]
        search_coords = amenity_coords_sampled
    else:
        search_coords = amenity_coords
    
    distances = []
    total_rentals = len(rental_coords)
    
    for i, rental_coord in enumerate(rental_coords):
        min_dist = min([geodesic(rental_coord, amenity_coord).meters 
                       for amenity_coord in search_coords])
        distances.append(min_dist)
        
        if (i + 1) % 5000 == 0:
            print(f"    ✅ Processed {i + 1:,}/{total_rentals:,} properties")
    
    return distances

print("🚀 STARTING OPTIMIZED DISTANCE CALCULATIONS")
print("=" * 60)
print(f"📊 Processing {len(combined_df_clean):,} rental properties")
print("=" * 60)

overall_start_time = time.time()
amenity_count = len(amenity_files)

'''
for amenity_name, filename in amenity_files.items():
    print(f"\n📍 [{list(amenity_files.keys()).index(amenity_name) + 1}/{amenity_count}] {amenity_name.upper()}")
    print("-" * 50)
    
    amenity_start_time = time.time()
    
    try:
        # Load amenity data
        amenity_df = pd.read_csv(filename)
        
        # Choose method based on amenity size
        if len(amenity_df) > 2000:  # Very large datasets
            print("  ⚡ Using OPTIMIZED method (very large dataset)")
            distances = calculate_distances_optimized(combined_df_clean, amenity_df, amenity_name)
        else:  # Small to medium datasets
            print("  🎯 Using SMART grid-based method")
            distances = calculate_distances_smart(combined_df_clean, amenity_df, amenity_name)
        
        # Add to dataframe
        combined_df_clean[f'distance_to_nearest_{amenity_name}'] = distances
        
        # Print summary
        amenity_time = time.time() - amenity_start_time
        mean_dist = np.mean(distances)
        min_dist = np.min(distances)
        max_dist = np.max(distances)
        
        print(f"  ✅ Completed in {amenity_time:.1f} seconds")
        print(f"  📏 Mean: {mean_dist:,.0f}m | Min: {min_dist:,.0f}m | Max: {max_dist:,.0f}m")
        
    except Exception as e:
        print(f"  ❌ Error: {e}")
        combined_df_clean[f'distance_to_nearest_{amenity_name}'] = np.nan

# Final summary
total_time = time.time() - overall_start_time
print(f"\n🎉 ALL COMPLETED in {total_time/60:.1f} minutes!")

# Save results
combined_df_clean.to_csv('rental_data_with_amenity_distances_OPTIMIZED.csv', index=False)
print("💾 Saved to 'rental_data_with_amenity_distances_OPTIMIZED.csv'")

# Show quick summary
distance_cols = [col for col in combined_df_clean.columns if 'distance_to_nearest' in col]
print(f"\n📊 Added {len(distance_cols)} distance features:")
for col in distance_cols:
    mean_val = combined_df_clean[col].mean()
    print(f"   {col}: {mean_val:,.0f}m average")
'''


In [ ]:
import pandas as pd
import numpy as np

# Load your enhanced dataset
print("📁 LOADING ENHANCED DATASET...")
df = pd.read_csv('Intermediate Files/rental_data_with_amenity_distances_OPTIMIZED.csv')
print(f"✅ Loaded dataset with {len(df):,} rows and {len(df.columns)} columns")

# Get all distance columns (the new features you just added)
distance_cols = [col for col in df.columns if 'distance_to_nearest' in col]
print(f"\n📍 FOUND {len(distance_cols)} DISTANCE FEATURES:")

# Check NaN values for each distance feature
print(f"\n🔍 CHECKING FOR NaN VALUES IN DISTANCE FEATURES:")
print("=" * 80)

nan_summary = []
for col in distance_cols:
    nan_count = df[col].isnull().sum()
    nan_percentage = (nan_count / len(df)) * 100
    nan_summary.append({
        'Feature': col,
        'NaN_Count': nan_count,
        'NaN_Percentage': f"{nan_percentage:.4f}%",
        'Data_Type': df[col].dtype,
        'Mean_Distance': f"{df[col].mean():.0f}m",
        'Min_Distance': f"{df[col].min():.0f}m",
        'Max_Distance': f"{df[col].max():.0f}m"
    })

# Create and display NaN summary
nan_df = pd.DataFrame(nan_summary)
print(nan_df.to_string(index=False))

# Show overall NaN statistics
total_nan_rows = df[distance_cols].isnull().any(axis=1).sum()
print(f"\n📊 OVERALL NaN SUMMARY:")
print(f"   • Properties with ANY NaN distance: {total_nan_rows:,} ({total_nan_rows/len(df)*100:.4f}%)")
print(f"   • Properties with ALL valid distances: {len(df) - total_nan_rows:,} ({(len(df) - total_nan_rows)/len(df)*100:.2f}%)")

# List all columns in the dataset with their data types
print(f"\n📋 COMPLETE COLUMN LIST WITH DATA TYPES:")
print("=" * 80)

# Categorize columns
original_cols = ['Project Name', 'Street Name', 'Postal District', 'Property Type',
                 'No of Bedroom', 'Monthly Rent ($)', 'Floor Area (SQM)',
                 'Floor Area (SQFT)', 'Lease Commencement Date', 'latitude', 
                 'longitude', 'geocode_status']

new_distance_cols = [col for col in df.columns if 'distance_to_nearest' in col]
other_cols = [col for col in df.columns if col not in original_cols and col not in new_distance_cols]

# Create comprehensive column summary
column_summary = []
for col in df.columns:
    dtype = df[col].dtype
    unique_count = df[col].nunique()
    null_count = df[col].isnull().sum()
    
    # Determine category
    if col in original_cols:
        category = "Original"
    elif col in new_distance_cols:
        category = "New Distance Feature"
    else:
        category = "Other"
    
    column_summary.append({
        'Column_Name': col,
        'Data_Type': dtype,
        'Category': category,
        'Unique_Values': unique_count,
        'Null_Values': null_count,
        'Null_Percentage': f"{(null_count/len(df))*100:.4f}%"
    })

# Display column summary
column_df = pd.DataFrame(column_summary)
print(column_df.to_string(index=False))

# Show statistics by category
print(f"\n📈 DATASET COMPOSITION BY CATEGORY:")
category_counts = column_df['Category'].value_counts()
for category, count in category_counts.items():
    print(f"   • {category}: {count} columns")

print(f"\n🎯 RECOMMENDED ACTIONS:")
# Check if any NaN handling is needed
if total_nan_rows > 0:
    print(f"   ⚠️  Need to handle NaN values in distance features")
    print(f"   💡 Options: Fill with median, mean, or large value (e.g., 10,000m)")
else:
    print(f"   ✅ No NaN handling needed - all distance features are complete")

# Check for duplicate/redundant features
if 'Floor Area (SQM)' in df.columns and 'Floor Area (SQFT)' in df.columns:
    print(f"   💡 Consider keeping only one floor area column (SQM/SQFT are correlated)")

print(f"\n💾 Your dataset is now ready with {len(new_distance_cols)} new distance features!")
print(f"   Total features available: {len(df.columns)}")
print(f"   Target variable: 'Monthly Rent ($)'")


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import re

def clean_rental_data(input_file='Intermediate Files/rental_data_with_amenity_distances_OPTIMIZED.csv'):
    """
    Clean and preprocess rental data for ML modeling
    """
    print("📁 LOADING ENHANCED DATASET...")
    df = pd.read_csv(input_file)
    print(f"✅ Loaded dataset with {len(df):,} rows and {len(df.columns)} columns")
    
    print(f"\n🔧 STEP 1: DATA CLEANING AND FEATURE CONVERSION")
    print("=" * 60)
    
    # Create a copy to avoid modifying original
    cleaned_df = df.copy()
    
    # 1. Handle 'No of Bedroom' - Drop due to high NaN percentage
    print("🗑️  Dropping 'No of Bedroom' (20.29% NaN values)")
    cleaned_df = cleaned_df.drop('No of Bedroom', axis=1)
    
    # 2. Convert Floor Area columns - they contain ranges like "100 to 110"
    print("📏 Converting Floor Area columns from ranges to numeric...")
    
    def convert_floor_area_range(area_str):
        """Convert range strings like '100 to 110' to numeric (take average)"""
        if pd.isna(area_str):
            return np.nan
        
        # Handle different formats
        if isinstance(area_str, (int, float)):
            return float(area_str)
        
        area_str = str(area_str).strip()
        
        # Remove commas (like "1,000 to 1,100")
        area_str = area_str.replace(',', '')
        
        # Try to extract numbers from range
        numbers = re.findall(r'\d+\.?\d*', area_str)
        
        if len(numbers) >= 2:
            # Take average of the range
            return (float(numbers[0]) + float(numbers[1])) / 2
        elif len(numbers) == 1:
            # Single number
            return float(numbers[0])
        else:
            return np.nan
    
    # Apply conversion to floor area columns
    cleaned_df['Floor_Area_SQM'] = cleaned_df['Floor Area (SQM)'].apply(convert_floor_area_range)
    cleaned_df['Floor_Area_SQFT'] = cleaned_df['Floor Area (SQFT)'].apply(convert_floor_area_range)
    
    # Check conversion results
    print(f"   Floor Area (SQM) sample conversion:")
    print(f"     Original: {cleaned_df['Floor Area (SQM)'].iloc[0]}")
    print(f"     Converted: {cleaned_df['Floor_Area_SQM'].iloc[0]:.1f}")
    print(f"   Floor Area (SQFT) sample conversion:")
    print(f"     Original: {cleaned_df['Floor Area (SQFT)'].iloc[0]}")
    print(f"     Converted: {cleaned_df['Floor_Area_SQFT'].iloc[0]:.1f}")
    
    # 3. Convert Lease Commencement Date and calculate months difference
    print("📅 Converting Lease Commencement Date and calculating months from now...")
    
    def convert_lease_date(date_str):
        """Convert dates like 'Jul-25' to proper datetime"""
        if pd.isna(date_str):
            return np.nan
        
        date_str = str(date_str).strip()
        
        # Handle "Jul-25" format (assuming 2025)
        if re.match(r'^[A-Za-z]{3}-\d{2}$', date_str):
            try:
                # Convert to datetime (assuming 2000s)
                return datetime.strptime(date_str, '%b-%y')
            except:
                return np.nan
        
        # Handle year-only format
        elif re.match(r'^\d{4}$', date_str):
            try:
                return datetime(int(date_str), 1, 1)
            except:
                return np.nan
        
        # Handle other date formats
        try:
            return pd.to_datetime(date_str)
        except:
            return np.nan
    
    # Convert to datetime
    cleaned_df['Lease_Commencement_Date'] = cleaned_df['Lease Commencement Date'].apply(convert_lease_date)
    
    # Calculate months difference from current date
    current_date = datetime.now()
    cleaned_df['Months_From_Now'] = ((current_date - cleaned_df['Lease_Commencement_Date']).dt.days / 30.44).round().astype(int)
    
    print(f"   Lease Date sample conversion:")
    print(f"     Original: {cleaned_df['Lease Commencement Date'].iloc[0]}")
    print(f"     Converted: {cleaned_df['Lease_Commencement_Date'].iloc[0]}")
    print(f"     Months from now: {cleaned_df['Months_From_Now'].iloc[0]}")
    
    # 4. Convert Postal District to categorical
    print("🏷️  Converting Postal District to categorical...")
    cleaned_df['Postal_District_Cat'] = cleaned_df['Postal District'].astype('category')
    print(f"   Postal District converted to category with {cleaned_df['Postal_District_Cat'].nunique()} unique values")
    
    # 5. Keep Project Name and Street Name as object (no conversion)
    print("📝 Keeping Project Name and Street Name as object type...")
    print(f"   Project Name unique values: {cleaned_df['Project Name'].nunique()}")
    print(f"   Street Name unique values: {cleaned_df['Street Name'].nunique()}")
    
    # 6. Handle any remaining NaN values
    print(f"\n🔍 Checking for remaining NaN values...")
    nan_summary = cleaned_df.isnull().sum()
    nan_columns = nan_summary[nan_summary > 0]
    
    if len(nan_columns) > 0:
        print("   Columns with NaN values:")
        for col, count in nan_columns.items():
            print(f"     {col}: {count} NaN ({count/len(cleaned_df)*100:.2f}%)")
        
        # Fill numerical columns with median
        numerical_cols = cleaned_df.select_dtypes(include=[np.number]).columns
        numerical_nan_cols = [col for col in nan_columns.index if col in numerical_cols]
        
        for col in numerical_nan_cols:
            median_val = cleaned_df[col].median()
            cleaned_df[col].fillna(median_val, inplace=True)
            print(f"     Filled {col} with median: {median_val:.2f}")
            
        # Fill categorical columns with mode
        categorical_cols = cleaned_df.select_dtypes(include=['object', 'category']).columns
        categorical_nan_cols = [col for col in nan_columns.index if col in categorical_cols]
        
        for col in categorical_nan_cols:
            if col in cleaned_df.columns:
                mode_val = cleaned_df[col].mode()[0] if len(cleaned_df[col].mode()) > 0 else 'Unknown'
                cleaned_df[col].fillna(mode_val, inplace=True)
                print(f"     Filled {col} with mode: '{mode_val}'")
    else:
        print("   ✅ No NaN values found!")
    
    print(f"\n🔧 STEP 2: FEATURE SELECTION")
    print("=" * 60)
    
    # Define features to drop (keep Project Name and Street Name as requested)
    features_to_drop = [
        'Floor Area (SQM)',       # Using converted numeric version
        'Floor Area (SQFT)',      # Using converted numeric version
        'Lease Commencement Date', # Using converted datetime version
        'Postal District',        # Using categorical version
        'Floor_Area_SQFT',        # Keep only SQM (they're correlated)
    ]
    
    ml_ready_df = cleaned_df.drop(features_to_drop, axis=1)
    
    print(f"   Dropped {len(features_to_drop)} features")
    print(f"   ML-ready dataset shape: {ml_ready_df.shape}")
    
    # Categorize final features
    categorical_features = ['Property Type', 'Postal_District_Cat', 'geocode_status', 'Project Name', 'Street Name']
    numerical_features = [col for col in ml_ready_df.columns 
                         if col not in categorical_features and 
                         col != 'Monthly Rent ($)' and 
                         col != 'Lease_Commencement_Date']
    
    datetime_features = ['Lease_Commencement_Date']
    
    print(f"\n📋 FINAL FEATURE SUMMARY:")
    print(f"   Total features: {len(ml_ready_df.columns)}")
    print(f"   Numerical features: {len(numerical_features)}")
    print(f"   Categorical features: {len(categorical_features)}")
    print(f"   Datetime features: {len(datetime_features)}")
    print(f"   Target variable: 'Monthly Rent ($)'")
    
    print(f"\n🔢 Numerical Features:")
    for feat in numerical_features:
        print(f"     • {feat}")
    
    print(f"\n📝 Categorical Features:")
    for feat in categorical_features:
        unique_vals = ml_ready_df[feat].nunique()
        print(f"     • {feat} ({unique_vals} unique values)")
    
    print(f"\n📅 Datetime Features:")
    for feat in datetime_features:
        print(f"     • {feat}")
    
    # Data quality report
    print(f"\n📊 DATA QUALITY REPORT:")
    print(f"   Total rows: {len(ml_ready_df):,}")
    print(f"   Total columns: {len(ml_ready_df.columns)}")
    print(f"   Memory usage: {ml_ready_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Check for any remaining issues
    remaining_nan = ml_ready_df.isnull().sum().sum()
    print(f"   Remaining NaN values: {remaining_nan}")
    
    # Show sample of cleaned data
    print(f"\n👀 SAMPLE OF CLEANED DATA:")
    sample_cols = ['Project Name', 'Street Name', 'Property Type', 'Postal_District_Cat', 
                   'Floor_Area_SQM', 'Months_From_Now', 'Monthly Rent ($)'] + numerical_features[:3]
    print(ml_ready_df[sample_cols].head().to_string())
    
    return ml_ready_df, numerical_features, categorical_features, datetime_features

def save_cleaned_data(ml_ready_df, numerical_features, categorical_features, datetime_features, output_file='rental_data_ML_ready.csv'):
    """
    Save the cleaned data and feature information
    """
    print(f"\n💾 SAVING CLEANED DATA")
    print("=" * 60)
    
    # Save the main cleaned dataset
    ml_ready_df.to_csv(output_file, index=False)
    print(f"✅ Cleaned data saved: {output_file}")
    
    # Save feature information
    all_features = numerical_features + categorical_features + datetime_features
    feature_types = (['numerical'] * len(numerical_features) + 
                    ['categorical'] * len(categorical_features) + 
                    ['datetime'] * len(datetime_features))
    
    feature_info = pd.DataFrame({
        'feature': all_features,
        'feature_type': feature_types,
        'dtype': [str(ml_ready_df[f].dtype) for f in all_features]
    })
    
    feature_info_file = 'feature_info.csv'
    feature_info.to_csv(feature_info_file, index=False)
    print(f"✅ Feature info saved: {feature_info_file}")
    
    # Save data summary
    summary = {
        'total_rows': len(ml_ready_df),
        'total_features': len(ml_ready_df.columns),
        'numerical_features': len(numerical_features),
        'categorical_features': len(categorical_features),
        'datetime_features': len(datetime_features),
        'target_variable': 'Monthly Rent ($)',
        'output_file': output_file
    }
    
    summary_df = pd.DataFrame([summary])
    summary_file = 'data_cleaning_summary.csv'
    summary_df.to_csv(summary_file, index=False)
    print(f"✅ Data summary saved: {summary_file}")  # Fixed this line
    
    print(f"\n🎉 DATA CLEANING COMPLETED!")
    print(f"   Your data is now ready for the next steps:")
    print(f"   1. Train-test splitting")
    print(f"   2. One-hot encoding categorical variables")
    print(f"   3. Scaling numerical features")
    print(f"   4. Model training!")

# Run the data cleaning
if __name__ == "__main__":
    # Clean the data
    ml_ready_df, numerical_features, categorical_features, datetime_features = clean_rental_data()
    
    # Save the results
    save_cleaned_data(ml_ready_df, numerical_features, categorical_features, datetime_features)


In [ ]:
import pandas as pd

# Load your cleaned dataset
df = pd.read_csv('rental_data_ML_ready.csv')

# Simple data type info
print("📋 COLUMN DATA TYPES:")
print("=" * 50)
print(df.dtypes)


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# Load your cleaned dataset
df = pd.read_csv('rental_data_ML_ready.csv')

print("🔧 FIXING DATA TYPES AND DROPPING COLUMNS")
print("=" * 60)

# 1. Drop geocode_status
print("🗑️  1. Dropping 'geocode_status' column...")
df = df.drop('geocode_status', axis=1)
print("   ✅ geocode_status dropped")

# 2. Convert Postal_District_Cat to categorical
print("\n🏷️  2. Converting 'Postal_District_Cat' to categorical...")
print(f"   Current unique values: {sorted(df['Postal_District_Cat'].unique())}")
df['Postal_District_Cat'] = df['Postal_District_Cat'].astype('category')
print("   ✅ Postal_District_Cat converted to categorical")

# 2. Convert Postal_District_Cat to categorical
print("\n🏷️  2. Converting 'Property Type' to categorical...")
print(f"   Current unique values: {sorted(df['Property Type'].unique())}")
df['Property Type'] = df['Property Type'].astype('category')
print("   ✅ Property Type converted to categorical")

# 3. Convert Lease_Commencement_Date to datetime
print("\n📅 3. Converting 'Lease_Commencement_Date' to datetime...")
print(f"   Sample before conversion: {df['Lease_Commencement_Date'].iloc[0]} (type: {type(df['Lease_Commencement_Date'].iloc[0])})")

# Convert to datetime
df['Lease_Commencement_Date'] = pd.to_datetime(df['Lease_Commencement_Date'])

print(f"   Sample after conversion: {df['Lease_Commencement_Date'].iloc[0]} (type: {type(df['Lease_Commencement_Date'].iloc[0])})")
print("   ✅ Lease_Commencement_Date converted to datetime")

# Verify the changes
print(f"\n✅ CHANGES COMPLETED!")
print(f"   Final dataset shape: {df.shape}")

# Show updated data types
print(f"\n📊 UPDATED DATA TYPES:")
print("=" * 50)
for col in df.columns:
    dtype = df[col].dtype
    null_count = df[col].isnull().sum()
    unique_count = df[col].nunique()
    
    print(f"{col:35} | {str(dtype):15} | Null: {null_count:4} | Unique: {unique_count:4}")

# Save the final cleaned dataset
output_file = 'rental_data_ML_ready_final.csv'
df.to_csv(output_file, index=False)
print(f"\n💾 Final cleaned dataset saved: {output_file}")


In [ ]:
import pandas as pd
import numpy as np

# Load your final dataset
df = pd.read_csv('rental_data_ML_ready_final.csv')

# Convert Lease_Commencement_Date to datetime if not already
df['Lease_Commencement_Date'] = pd.to_datetime(df['Lease_Commencement_Date'])

print("📅 SPLITTING DATA BY QUARTERS (80% train, 20% test)")
print("=" * 60)

# Extract quarter and year from the lease date
df['Year'] = df['Lease_Commencement_Date'].dt.year
df['Quarter'] = df['Lease_Commencement_Date'].dt.quarter

print("📊 Data distribution by year and quarter:")
quarter_counts = df.groupby(['Year', 'Quarter']).size().reset_index(name='Count')
print(quarter_counts.to_string(index=False))

# Initialize empty dataframes for training and testing
train_dfs = []
test_dfs = []

# Get unique year-quarter combinations
unique_quarters = df[['Year', 'Quarter']].drop_duplicates().sort_values(['Year', 'Quarter'])

print(f"\n🔀 Splitting each quarter into 80% train / 20% test...")

for _, (year, quarter) in unique_quarters.iterrows():
    # Get data for this specific quarter
    quarter_data = df[(df['Year'] == year) & (df['Quarter'] == quarter)]
    
    if len(quarter_data) > 1:  # Need at least 2 rows to split
        # Calculate split sizes
        train_size = int(0.8 * len(quarter_data))
        
        # Split the data (random state for reproducibility)
        train_quarter = quarter_data.sample(n=train_size, random_state=42)
        test_quarter = quarter_data.drop(train_quarter.index)
        
        train_dfs.append(train_quarter)
        test_dfs.append(test_quarter)
        
        print(f"   {year}-Q{quarter}: {len(quarter_data):,} rows → Train: {len(train_quarter):,} | Test: {len(test_quarter):,}")
    else:
        print(f"   {year}-Q{quarter}: {len(quarter_data):,} rows → SKIPPED (insufficient data)")

# Combine all quarters
train_final = pd.concat(train_dfs, ignore_index=True)
test_final = pd.concat(test_dfs, ignore_index=True)

# Remove the temporary Year and Quarter columns
train_final = train_final.drop(['Year', 'Quarter'], axis=1)
test_final = test_final.drop(['Year', 'Quarter'], axis=1)

print(f"\n📊 FINAL SPLIT SUMMARY:")
print("=" * 40)
print(f"Training set: {len(train_final):,} rows ({len(train_final)/len(df)*100:.1f}%)")
print(f"Testing set:  {len(test_final):,} rows ({len(test_final)/len(df)*100:.1f}%)")
print(f"Total:        {len(train_final) + len(test_final):,} rows")

print(f"\n🎯 DATA QUALITY CHECK:")
print(f"Training set columns: {len(train_final.columns)}")
print(f"Testing set columns:  {len(test_final.columns)}")
print(f"Training target stats - Mean: ${train_final['Monthly Rent ($)'].mean():.2f}")
print(f"Testing target stats  - Mean: ${test_final['Monthly Rent ($)'].mean():.2f}")

# Check for any data leakage
train_dates = train_final['Lease_Commencement_Date']
test_dates = test_final['Lease_Commencement_Date']

print(f"\n📅 DATE RANGE CHECK:")
print(f"Training date range: {train_dates.min()} to {train_dates.max()}")
print(f"Testing date range:  {test_dates.min()} to {test_dates.max()}")

# Save the final train/test datasets
print(f"\n💾 SAVING TRAIN/TEST DATASETS")
print("=" * 40)
train_final.to_csv('rental_train_final.csv', index=False)
test_final.to_csv('rental_test_final.csv', index=False)

print("✅ Training data saved: rental_train_final.csv")
print("✅ Testing data saved: rental_test_final.csv")

# Show sample of both datasets
print(f"\n👀 TRAINING DATA SAMPLE:")
print(train_final[['Lease_Commencement_Date', 'Property Type', 'Monthly Rent ($)']].head().to_string())
print(f"\n👀 TESTING DATA SAMPLE:")
print(test_final[['Lease_Commencement_Date', 'Property Type', 'Monthly Rent ($)']].head().to_string())


In [ ]:
# FINAL FINAL!

# FINAL ONE - WITH TOP 5 FEATURES AND OVERFITTING ANALYSIS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

# Set colorful style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Load the datasets
print("📁 LOADING TRAIN/TEST DATASETS...")
train_df = pd.read_csv('rental_train_final.csv')
test_df = pd.read_csv('rental_test_final.csv')

print(f"Training set: {train_df.shape}")
print(f"Testing set: {test_df.shape}")

# Convert categorical columns (ONLY Property Type now, drop Postal District)
print("\n🔧 PREPROCESSING DATA...")
categorical_cols = ['Property Type']  # Only keep Property Type, drop Postal District
for col in categorical_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')

# Convert datetime to numeric (days since min date) - KEEP BOTH TEMPORAL FEATURES
train_df['Lease_Commencement_Date'] = pd.to_datetime(train_df['Lease_Commencement_Date'])
test_df['Lease_Commencement_Date'] = pd.to_datetime(test_df['Lease_Commencement_Date'])

min_date = train_df['Lease_Commencement_Date'].min()
train_df['Days_Since_Min'] = (train_df['Lease_Commencement_Date'] - min_date).dt.days
test_df['Days_Since_Min'] = (test_df['Lease_Commencement_Date'] - min_date).dt.days

# Define features - DROP COORDINATES AND POSTAL DISTRICT, KEEP TEMPORAL FEATURES
features_to_drop = ['Project Name', 'Street Name', 'Monthly Rent ($)', 'Lease_Commencement_Date', 
                   'latitude', 'longitude', 'Postal_District_Cat']  # ADDED Postal_District_Cat to drop

feature_columns = [col for col in train_df.columns if col not in features_to_drop]

print(f"✅ Features used: {len(feature_columns)}")
print(f"✅ Temporal features KEPT: Months_From_Now, Days_Since_Min")
print(f"✅ Postal District: DROPPED")
print(f"✅ Features dropped: {[col for col in features_to_drop if col in train_df.columns]}")

# Prepare data
X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()
y_train = train_df['Monthly Rent ($)']
y_test = test_df['Monthly Rent ($)']

# Handle categorical features for models that need encoding
categorical_features = ['Property Type']  # Only Property Type remains
numerical_features = [col for col in feature_columns if col not in categorical_features]

print(f"\n📊 FINAL FEATURE BREAKDOWN:")
print(f"  Categorical features: {categorical_features}")
print(f"  Numerical features: {len(numerical_features)}")
print(f"  Temporal features: Months_From_Now, Days_Since_Min")
print(f"  Amenity distance features: {len([f for f in numerical_features if 'distance' in f])}")

# One-hot encode categorical features for Linear Regression
X_train_encoded = pd.get_dummies(X_train, columns=categorical_features, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_features, drop_first=True)

# Ensure both train and test have same columns after encoding
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# Scale numerical features for Linear Regression
scaler = StandardScaler()
X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()

X_train_scaled[numerical_features] = scaler.fit_transform(X_train_encoded[numerical_features])
X_test_scaled[numerical_features] = scaler.transform(X_test_encoded[numerical_features])

print(f"\n🎯 TRAINING MODELS...")
print("=" * 60)

# Dictionary to store results
models = {}
feature_importances = {}
train_predictions = {}
test_predictions = {}

# 1. LINEAR REGRESSION
print("1. Training Linear Regression...")
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
train_predictions['Linear Regression'] = lr.predict(X_train_scaled)
test_predictions['Linear Regression'] = lr.predict(X_test_scaled)
models['Linear Regression'] = lr
feature_importances['Linear Regression'] = dict(zip(X_train_scaled.columns, np.abs(lr.coef_)))

# 2. RANDOM FOREST
print("2. Training Random Forest...")
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_encoded, y_train)
train_predictions['Random Forest'] = rf.predict(X_train_encoded)
test_predictions['Random Forest'] = rf.predict(X_test_encoded)
models['Random Forest'] = rf
feature_importances['Random Forest'] = dict(zip(X_train_encoded.columns, rf.feature_importances_))

# 3. CATBOOST
print("3. Training CatBoost...")
cat_features_indices = [X_train.columns.get_loc(col) for col in categorical_features if col in X_train.columns]
cb = CatBoostRegressor(iterations=1000, learning_rate=0.1, depth=6, random_state=42, verbose=False)
cb.fit(X_train, y_train, cat_features=cat_features_indices)
train_predictions['CatBoost'] = cb.predict(X_train)
test_predictions['CatBoost'] = cb.predict(X_test)
models['CatBoost'] = cb
feature_importances['CatBoost'] = dict(zip(X_train.columns, cb.get_feature_importance()))

# 4. XGBOOST
print("4. Training XGBoost...")
X_train_xgb = pd.get_dummies(X_train, columns=categorical_features)
X_test_xgb = pd.get_dummies(X_test, columns=categorical_features)
X_test_xgb = X_test_xgb.reindex(columns=X_train_xgb.columns, fill_value=0)
xgb_model = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1)
xgb_model.fit(X_train_xgb, y_train)
train_predictions['XGBoost'] = xgb_model.predict(X_train_xgb)
test_predictions['XGBoost'] = xgb_model.predict(X_test_xgb)
models['XGBoost'] = xgb_model
feature_importances['XGBoost'] = dict(zip(X_train_xgb.columns, xgb_model.feature_importances_))

print("✅ All models trained successfully!")

# Calculate metrics for all models
print(f"\n📊 MODEL PERFORMANCE COMPARISON")
print("=" * 60)

performance_metrics = []

for model_name in models.keys():
    # Test metrics
    y_pred_test = test_predictions[model_name]
    test_mae = mean_absolute_error(y_test, y_pred_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    test_r2 = r2_score(y_test, y_pred_test)
    
    # Train metrics
    y_pred_train = train_predictions[model_name]
    train_r2 = r2_score(y_train, y_pred_train)
    
    performance_metrics.append({
        'Model': model_name,
        'Train_R²': train_r2,
        'Test_R²': test_r2,
        'R²_Difference': train_r2 - test_r2,
        'MAE': test_mae,
        'RMSE': test_rmse
    })
    
    print(f"\n{model_name}:")
    print(f"  Train R²: {train_r2:.4f}")
    print(f"  Test R²:  {test_r2:.4f}")
    print(f"  Difference: {train_r2 - test_r2:.4f}")
    print(f"  MAE: ${test_mae:,.2f}")
    print(f"  RMSE: ${test_rmse:,.2f}")

# Create performance comparison table
performance_df = pd.DataFrame(performance_metrics)
print(f"\n🎯 PERFORMANCE COMPARISON TABLE:")
print("=" * 60)
print(performance_df.round(4).to_string(index=False))

# Save performance table
performance_df.to_csv('model_performance_no_postal.csv', index=False)
print("✅ Performance comparison saved to 'model_performance_no_postal.csv'")

# FEATURE IMPORTANCE VISUALIZATION - TOP 5 ONLY WITH SIMPLIFIED NAMES
print(f"\n📈 FEATURE IMPORTANCE ANALYSIS - TOP 5 FEATURES")
print("=" * 60)

# Function to simplify feature names for presentation
def simplify_feature_name(feature_name):
    name_map = {
        'distance_to_nearest_schools': 'Schools',
        'distance_to_nearest_mrt_stations': 'MRT Stations',
        'distance_to_nearest_supermarkets': 'Supermarkets',
        'distance_to_nearest_parks': 'Parks',
        'distance_to_nearest_taxi_stops': 'Taxi Stops',
        'distance_to_nearest_gyms': 'Gyms',
        'distance_to_nearest_hawker_centres': 'Hawker Centres',
        'distance_to_nearest_cycling_paths': 'Cycling Paths',
        'distance_to_nearest_tourist_attractions': 'Tourist Spots',
        'distance_to_nearest_disability_services': 'Disability Services',
        'Floor_Area_SQM': 'Floor Area',
        'Months_From_Now': 'Property Age',
        'Days_Since_Min': 'Market Time',
        'Property Type_Condominium': 'Condo Type',
        'Property Type_Executive Condominium': 'Exec Condo',
        'Property Type_Apartment': 'Apartment Type',
        'Property Type_HDB': 'HDB Type'
    }
    return name_map.get(feature_name, feature_name)

# Create feature importance plots for each model - TOP 5 ONLY
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

# Define a professional color palette
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3E885B']

for idx, (model_name, importance_dict) in enumerate(feature_importances.items()):
    # Get top 5 features only
    importance_df = pd.DataFrame({
        'feature': list(importance_dict.keys()),
        'importance': list(importance_dict.values())
    }).sort_values('importance', ascending=False).head(5)
    
    # Simplify feature names for presentation
    importance_df['feature_simple'] = importance_df['feature'].apply(simplify_feature_name)
    
    # Create colorful horizontal bar plot
    bars = axes[idx].barh(importance_df['feature_simple'], importance_df['importance'], 
                         color=colors, edgecolor='black', alpha=0.8, height=0.6)
    
    axes[idx].set_title(f'{model_name}', fontsize=14, fontweight='bold', pad=15)
    axes[idx].set_xlabel('Importance', fontsize=11, fontweight='bold')
    axes[idx].tick_params(axis='y', labelsize=10)
    axes[idx].grid(axis='x', alpha=0.3)
    
    # Add value labels on bars
    for i, (v, bar) in enumerate(zip(importance_df['importance'], bars)):
        axes[idx].text(bar.get_width() + bar.get_width()*0.01, bar.get_y() + bar.get_height()/2, 
                      f'{v:.3f}', va='center', ha='left', fontsize=9, fontweight='bold')

plt.suptitle('Top 5 Most Important Features by Model', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('feature_importance_top5_simple.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Combined feature importance comparison (top 5 features across all models)
print(f"\n🔍 COMBINED FEATURE IMPORTANCE SUMMARY - TOP 5")
print("=" * 50)

# Create a combined feature importance dataframe
all_features_set = set()
for importance_dict in feature_importances.values():
    all_features_set.update(importance_dict.keys())

combined_importance = pd.DataFrame(index=list(all_features_set))

for model_name, importance_dict in feature_importances.items():
    combined_importance[model_name] = combined_importance.index.map(importance_dict).fillna(0)

# Get average importance
combined_importance['Average'] = combined_importance.mean(axis=1)
top_features_combined = combined_importance.nlargest(5, 'Average')

# Simplify names for combined chart
top_features_combined['Simple_Name'] = [simplify_feature_name(idx) for idx in top_features_combined.index]

print("\nTop 5 Most Important Features (Average across all models):")
print(top_features_combined[['Average', 'Simple_Name']].sort_values('Average', ascending=True))

# Plot combined feature importance - TOP 5
plt.figure(figsize=(12, 8))
top_5_sorted = top_features_combined['Average'].sort_values(ascending=True)
simple_names = [simplify_feature_name(idx) for idx in top_5_sorted.index]

# Create professional color palette
colors_combined = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3E885B']

bars = plt.barh(range(len(top_5_sorted)), top_5_sorted.values, color=colors_combined, 
                edgecolor='black', alpha=0.8, height=0.6)

plt.yticks(range(len(top_5_sorted)), simple_names, fontsize=12)
plt.xlabel('Average Importance Score', fontsize=13, fontweight='bold')
plt.title('Top 5 Most Important Features\n(Average Across All 4 ML Models)', 
          fontsize=15, fontweight='bold', pad=20)

# Add value labels
for i, (bar, value) in enumerate(zip(bars, top_5_sorted.values)):
    plt.text(bar.get_width() + bar.get_width()*0.01, bar.get_y() + bar.get_height()/2, 
             f'{value:.4f}', va='center', ha='left', fontsize=11, fontweight='bold')

plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('combined_feature_importance_top5.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# OVERFITTING ANALYSIS CHART
print(f"\n🚨 OVERFITTING ANALYSIS")
print("=" * 50)

# Create overfitting analysis chart
plt.figure(figsize=(12, 8))

models_list = list(models.keys())
train_r2_scores = [r2_score(y_train, train_predictions[model]) for model in models_list]
test_r2_scores = [r2_score(y_test, test_predictions[model]) for model in models_list]
r2_differences = [train_r2 - test_r2 for train_r2, test_r2 in zip(train_r2_scores, test_r2_scores)]

x = np.arange(len(models_list))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 8))
bars1 = ax.bar(x - width/2, train_r2_scores, width, label='Train R²', color='#2E86AB', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x + width/2, test_r2_scores, width, label='Test R²', color='#F18F01', alpha=0.8, edgecolor='black')

# Add difference values on top
for i, (train_r2, test_r2, diff) in enumerate(zip(train_r2_scores, test_r2_scores, r2_differences)):
    ax.text(i, max(train_r2, test_r2) + 0.02, f'Δ={diff:.3f}', 
            ha='center', va='bottom', fontsize=10, fontweight='bold', color='red')

ax.set_xlabel('Machine Learning Models', fontsize=13, fontweight='bold')
ax.set_ylabel('R² Score', fontsize=13, fontweight='bold')
ax.set_title('Overfitting Analysis: Train vs Test Performance\n(R² Score Comparison)', 
             fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(models_list, fontsize=11)
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)

# Add overfitting warnings
overfitting_threshold = 0.1  # If train R² is more than 0.1 higher than test R²
for i, diff in enumerate(r2_differences):
    if diff > overfitting_threshold:
        ax.text(i, 0.5, '⚠️ Overfitting', ha='center', va='center', 
                fontsize=11, fontweight='bold', color='red', rotation=45,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))

plt.tight_layout()
plt.savefig('overfitting_analysis.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Overfitting analysis summary
print(f"\n🔍 OVERFITTING ANALYSIS SUMMARY:")
print("=" * 40)
for i, model_name in enumerate(models_list):
    diff = r2_differences[i]
    status = "⚠️  OVERFITTING" if diff > 0.1 else "✅ Good Generalization"
    print(f"{model_name:20} | Train R²: {train_r2_scores[i]:.4f} | Test R²: {test_r2_scores[i]:.4f} | Diff: {diff:.4f} | {status}")

print(f"\n📝 INTERPRETATION:")
print("• R² Difference > 0.10: Potential overfitting")
print("• R² Difference < 0.05: Good generalization")
print("• Test R² is the true measure of model performance")

print(f"\n🎉 MODEL TRAINING COMPLETED!")
print("=" * 50)
print("📁 Generated Files:")
print("  - model_performance_no_postal.csv")
print("  - feature_importance_top5_simple.png")
print("  - combined_feature_importance_top5.png")
print("  - overfitting_analysis.png")

# Show best performing model
best_model = performance_df.loc[performance_df['Test_R²'].idxmax()]
print(f"\n🏆 BEST PERFORMING MODEL: {best_model['Model']}")
print(f"   Test R²: {best_model['Test_R²']:.4f}")
print(f"   MAE: ${best_model['MAE']:,.2f}")
print(f"   RMSE: ${best_model['RMSE']:,.2f}")
print(f"   Overfitting Risk: {'HIGH ⚠️' if best_model['R²_Difference'] > 0.1 else 'LOW ✅'}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("🎯 ANALYZING VALUE SCORES BY POSTAL DISTRICT")
print("=" * 60)

# Load the datasets
train_df = pd.read_csv('rental_train_final.csv')
test_df = pd.read_csv('rental_test_final.csv')

print(f"Test dataset shape: {test_df.shape}")

# Convert datetime columns and recreate Days_Since_Min using the same logic
print("🔧 Recreating temporal features...")
train_df['Lease_Commencement_Date'] = pd.to_datetime(train_df['Lease_Commencement_Date'])
test_df['Lease_Commencement_Date'] = pd.to_datetime(test_df['Lease_Commencement_Date'])

# Use the SAME min_date from training data to ensure consistency
min_date = train_df['Lease_Commencement_Date'].min()
train_df['Days_Since_Min'] = (train_df['Lease_Commencement_Date'] - min_date).dt.days
test_df['Days_Since_Min'] = (test_df['Lease_Commencement_Date'] - min_date).dt.days

print(f"✅ Recreated Days_Since_Min using min_date: {min_date}")

# Convert Postal_District_Cat to integer to fix the formatting error
test_df['Postal_District_Cat'] = test_df['Postal_District_Cat'].astype(int)
train_df['Postal_District_Cat'] = train_df['Postal_District_Cat'].astype(int)

print("✅ Converted Postal_District_Cat to integer")

# Define features based on what should be available
categorical_features = ['Property Type', 'Postal_District_Cat']

# Get all distance features that are available
distance_features = [col for col in test_df.columns if 'distance_to_nearest' in col]
numerical_features = ['Floor_Area_SQM', 'Months_From_Now', 'Days_Since_Min'] + distance_features

print(f"✅ Using {len(numerical_features)} numerical features")
print(f"✅ Using {len(categorical_features)} categorical features")

# Prepare test data
X_test = test_df[numerical_features + categorical_features].copy()

# One-hot encode categorical features for XGBoost
X_test_encoded = pd.get_dummies(X_test, columns=categorical_features)

# Load the trained XGBoost model (retrain if needed)
print("🔧 Preparing XGBoost model...")
from xgboost import XGBRegressor

# Prepare training data
X_train = train_df[numerical_features + categorical_features].copy()
y_train = train_df['Monthly Rent ($)']
X_train_encoded = pd.get_dummies(X_train, columns=categorical_features)

# Ensure test data has same columns as training data
missing_cols = set(X_train_encoded.columns) - set(X_test_encoded.columns)
for col in missing_cols:
    X_test_encoded[col] = 0
X_test_encoded = X_test_encoded[X_train_encoded.columns]

# Train XGBoost model
xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_encoded, y_train)
print("✅ XGBoost model trained successfully!")

# Make predictions on test data
print("📊 Making predictions on test data...")
y_pred = xgb_model.predict(X_test_encoded)

# Add predictions and value scores to test dataframe
test_df['Predicted_Rent'] = y_pred

# CALCULATE VALUE SCORE (MULTIPLIED BY 100 AS REQUESTED)
test_df['Value_Score'] = (test_df['Monthly Rent ($)'] / test_df['Predicted_Rent']) * 100

print(f"\n📈 VALUE SCORE STATISTICS (Scale: 100 = Fair Value):")
print(f"  Mean Value Score: {test_df['Value_Score'].mean():.1f}")
print(f"  Median Value Score: {test_df['Value_Score'].median():.1f}")
print(f"  Std Value Score: {test_df['Value_Score'].std():.1f}")
print(f"  Min Value Score: {test_df['Value_Score'].min():.1f}")
print(f"  Max Value Score: {test_df['Value_Score'].max():.1f}")

# Calculate average value score by postal district
print(f"\n🏷️  ANALYZING BY POSTAL DISTRICT...")
postal_value_scores = test_df.groupby('Postal_District_Cat').agg({
    'Value_Score': ['mean', 'count', 'std'],
    'Monthly Rent ($)': 'mean',
    'Predicted_Rent': 'mean'
}).round(1)

# Flatten column names
postal_value_scores.columns = ['Value_Score_Mean', 'Property_Count', 'Value_Score_Std', 
                              'Actual_Rent_Mean', 'Predicted_Rent_Mean']

# Reset index for easier handling
postal_value_scores = postal_value_scores.reset_index()

# Filter districts with sufficient data (at least 10 properties)
postal_value_scores = postal_value_scores[postal_value_scores['Property_Count'] >= 10]

print(f"Districts with sufficient data: {len(postal_value_scores)}")

# Sort by value score
postal_value_scores_sorted = postal_value_scores.sort_values('Value_Score_Mean', ascending=False)

# Get top 3 and bottom 3 districts
top_3_districts = postal_value_scores_sorted.head(3)
bottom_3_districts = postal_value_scores_sorted.tail(3)

print(f"\n🏆 TOP 3 BEST VALUE DISTRICTS (Undervalued - Green):")
print("=" * 60)
for _, row in top_3_districts.iterrows():
    percentage_diff = ((row['Value_Score_Mean'] - 100) / 100 * 100)
    print(f"District {int(row['Postal_District_Cat']):2d}:")  # Convert to int for formatting
    print(f"  Value Score: {row['Value_Score_Mean']:.1f} (Higher = Better Value)")
    print(f"  Properties: {int(row['Property_Count']):3d}")  # Convert to int
    print(f"  Avg Actual Rent: ${row['Actual_Rent_Mean']:,.0f}")
    print(f"  Avg Predicted Rent: ${row['Predicted_Rent_Mean']:,.0f}")
    print(f"  Interpretation: Rents are {percentage_diff:+.1f}% {'lower' if percentage_diff > 0 else 'higher'} than predicted")
    print()

print(f"\n🔻 BOTTOM 3 WORST VALUE DISTRICTS (Overvalued - Red):")
print("=" * 60)
for _, row in bottom_3_districts.iterrows():
    percentage_diff = ((row['Value_Score_Mean'] - 100) / 100 * 100)
    print(f"District {int(row['Postal_District_Cat']):2d}:")  # Convert to int for formatting
    print(f"  Value Score: {row['Value_Score_Mean']:.1f} (Lower = Worse Value)")
    print(f"  Properties: {int(row['Property_Count']):3d}")  # Convert to int
    print(f"  Avg Actual Rent: ${row['Actual_Rent_Mean']:,.0f}")
    print(f"  Avg Predicted Rent: ${row['Predicted_Rent_Mean']:,.0f}")
    print(f"  Interpretation: Rents are {percentage_diff:+.1f}% {'lower' if percentage_diff > 0 else 'higher'} than predicted")
    print()

# Create visualization
print(f"\n📊 CREATING VALUE SCORE VISUALIZATION...")
plt.figure(figsize=(14, 10))

# Combine top and bottom districts for plotting
highlight_districts = pd.concat([top_3_districts, bottom_3_districts])

# Create color mapping
colors = []
for district in highlight_districts['Postal_District_Cat']:
    if district in top_3_districts['Postal_District_Cat'].values:
        colors.append('#2E8B57')  # Green for top 3
    else:
        colors.append('#DC143C')  # Red for bottom 3

# Create the bar plot
bars = plt.bar(range(len(highlight_districts)), 
               highlight_districts['Value_Score_Mean'], 
               color=colors, alpha=0.8, edgecolor='black')

plt.xlabel('Postal District', fontsize=13, fontweight='bold')
plt.ylabel('Value Score × 100\n(Actual / Predicted Rent × 100)', fontsize=13, fontweight='bold')
plt.title('Top 3 (Green) vs Bottom 3 (Red) Postal Districts by Value Score\n(Scale: 100 = Fair Value)', 
          fontsize=15, fontweight='bold', pad=20)

# Set x-axis labels with district numbers
plt.xticks(range(len(highlight_districts)), 
           [f"District {int(d)}" for d in highlight_districts['Postal_District_Cat']],  # Convert to int
           rotation=45, fontsize=11)

# Add value labels on bars
for i, (bar, score, count) in enumerate(zip(bars, highlight_districts['Value_Score_Mean'], highlight_districts['Property_Count'])):
    percentage_diff = ((score - 100) / 100 * 100)
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'{score:.1f}\n({percentage_diff:+.1f}%)', 
             ha='center', va='bottom', fontsize=10, fontweight='bold')

# Add reference line at 100 (fair value)
plt.axhline(y=100, color='black', linestyle='--', alpha=0.7, linewidth=2)
plt.text(len(highlight_districts)-0.5, 102, 'Fair Value (100)', 
         ha='right', va='bottom', fontsize=10, fontweight='bold', color='black')

# Add interpretation annotations
plt.text(0.5, plt.ylim()[1] * 0.85, '🎯 VALUE INTERPRETATION:\n• Score > 100 = Undervalued (Good Buy)\n• Score < 100 = Overvalued (Expensive)', 
         bbox=dict(boxstyle="round,pad=0.5", facecolor="lightblue", alpha=0.8),
         fontsize=11, fontweight='bold')

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('postal_district_value_scores_x100.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Create detailed table for all districts
print(f"\n📋 DETAILED VALUE SCORES FOR ALL DISTRICTS (Scale: 100 = Fair Value):")
print("=" * 80)
all_districts_sorted = postal_value_scores.sort_values('Value_Score_Mean', ascending=False)

# Color code the output in console
for _, row in all_districts_sorted.iterrows():
    district = int(row['Postal_District_Cat'])  # Convert to int
    score = row['Value_Score_Mean']
    count = int(row['Property_Count'])  # Convert to int
    actual_rent = row['Actual_Rent_Mean']
    predicted_rent = row['Predicted_Rent_Mean']
    percentage_diff = ((score - 100) / 100 * 100)
    
    if district in [int(x) for x in top_3_districts['Postal_District_Cat'].values]:
        color_indicator = "🟢 TOP 3"
    elif district in [int(x) for x in bottom_3_districts['Postal_District_Cat'].values]:
        color_indicator = "🔴 BOTTOM 3"
    else:
        color_indicator = "⚪"
    
    print(f"{color_indicator} District {district:2d}: Score={score:5.1f} | "
          f"Props={count:3d} | Actual=${actual_rent:,.0f} | Predicted=${predicted_rent:,.0f} | "
          f"Diff={percentage_diff:+.1f}%")

# Save detailed results to CSV
detailed_results = all_districts_sorted.copy()
detailed_results['Percentage_Difference'] = (detailed_results['Value_Score_Mean'] - 100) / 100 * 100
detailed_results['Value_Category'] = detailed_results['Postal_District_Cat'].apply(
    lambda x: 'Top 3' if x in top_3_districts['Postal_District_Cat'].values else 
              'Bottom 3' if x in bottom_3_districts['Postal_District_Cat'].values else 'Middle'
)

detailed_results.to_csv('postal_district_value_analysis_x100.csv', index=False)
print(f"\n💾 Detailed analysis saved to 'postal_district_value_analysis_x100.csv'")

print(f"\n🎯 BUSINESS INSIGHTS:")
print("=" * 50)
print("🏆 TOP DISTRICTS (Green): Good opportunities for tenants/renters")
print("   • Value Score > 100: Rents are lower than predicted")
print("   • Potentially undervalued areas with good amenities")

print("\n🔻 BOTTOM DISTRICTS (Red): Premium pricing areas")
print("   • Value Score < 100: Rents are higher than predicted")
print("   • Could be due to prestige, location premium, or other unmeasured factors")
print("   • Good for landlords, expensive for tenants")

print(f"\n📊 VALUE SCORE SCALE:")
print("   • 100 = Fair value (rent matches prediction)")
print("   • > 100 = Undervalued (good deal for renters)")
print("   • < 100 = Overvalued (expensive for renters)")

print(f"\n✅ ANALYSIS COMPLETED!")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("🎯 ANALYZING VALUE SCORES BY POSTAL DISTRICT")
print("=" * 60)
print("MODEL FEATURES: 10 distances + Floor Area + 2 temporal + Property Type")
print("POSTAL DISTRICT: Used only for grouping results (NOT in model)")

# Load the datasets
train_df = pd.read_csv('rental_train_final.csv')
test_df = pd.read_csv('rental_test_final.csv')

print(f"Test dataset shape: {test_df.shape}")

# Convert datetime columns and recreate Days_Since_Min using the same logic
print("🔧 Recreating temporal features...")
train_df['Lease_Commencement_Date'] = pd.to_datetime(train_df['Lease_Commencement_Date'])
test_df['Lease_Commencement_Date'] = pd.to_datetime(test_df['Lease_Commencement_Date'])

# Use the SAME min_date from training data to ensure consistency
min_date = train_df['Lease_Commencement_Date'].min()
train_df['Days_Since_Min'] = (train_df['Lease_Commencement_Date'] - min_date).dt.days
test_df['Days_Since_Min'] = (test_df['Lease_Commencement_Date'] - min_date).dt.days

# Convert Postal_District_Cat to integer
test_df['Postal_District_Cat'] = test_df['Postal_District_Cat'].astype(int)
train_df['Postal_District_Cat'] = train_df['Postal_District_Cat'].astype(int)

# ✅ CORRECT: Define features WITHOUT postal district for the model
categorical_features = ['Property Type']  # Only Property Type, NO Postal District
distance_features = [col for col in test_df.columns if 'distance_to_nearest' in col]
numerical_features = ['Floor_Area_SQM', 'Months_From_Now', 'Days_Since_Min'] + distance_features

print(f"\n📋 MODEL FEATURES USED FOR PREDICTION:")
print(f"  Categorical: {categorical_features}")
print(f"  Numerical: {len(numerical_features)} features")
print(f"    - Floor Area: 1 feature")
print(f"    - Temporal: 2 features (Months_From_Now, Days_Since_Min)")
print(f"    - Distance features: {len(distance_features)} features")
print(f"  TOTAL FEATURES IN MODEL: {len(numerical_features) + len(categorical_features)}")
print(f"  POSTAL DISTRICT: Used ONLY for grouping results (not in model)")

# Prepare data and train model WITHOUT postal district
from xgboost import XGBRegressor

# Prepare training data WITHOUT postal district
X_train = train_df[numerical_features + categorical_features].copy()
y_train = train_df['Monthly Rent ($)']
X_train_encoded = pd.get_dummies(X_train, columns=categorical_features)

# Prepare test data WITHOUT postal district
X_test = test_df[numerical_features + categorical_features].copy()
X_test_encoded = pd.get_dummies(X_test, columns=categorical_features)

# Ensure test data has same columns as training data
missing_cols = set(X_train_encoded.columns) - set(X_test_encoded.columns)
for col in missing_cols:
    X_test_encoded[col] = 0
X_test_encoded = X_test_encoded[X_train_encoded.columns]

# Train XGBoost model WITHOUT postal district features
xgb_model = XGBRegressor(n_estimators=1000, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1)
xgb_model.fit(X_train_encoded, y_train)
print("✅ XGBoost model trained successfully WITHOUT postal district features!")

# Make predictions
y_pred = xgb_model.predict(X_test_encoded)
test_df['Predicted_Rent'] = y_pred

# ✅ STEP 1: Calculate INDIVIDUAL value scores for each property
test_df['Value_Score'] = (test_df['Monthly Rent ($)'] / test_df['Predicted_Rent']) * 100

print(f"\n🔍 VERIFICATION: INDIVIDUAL VALUE SCORES CALCULATION")
print("=" * 50)
print(f"Total properties with individual value scores: {len(test_df)}")

# ✅ STEP 2: Use postal district ONLY for grouping (not in model)
print(f"\n🏷️  GROUPING RESULTS BY POSTAL DISTRICT (NOT in model)")
print("=" * 50)

postal_value_scores = test_df.groupby('Postal_District_Cat').agg({
    'Value_Score': ['mean', 'count', 'std'],
    'Monthly Rent ($)': 'mean',
    'Predicted_Rent': 'mean'
}).round(1)

# Flatten column names
postal_value_scores.columns = ['Value_Score_Mean', 'Property_Count', 'Value_Score_Std', 
                              'Actual_Rent_Mean', 'Predicted_Rent_Mean']
postal_value_scores = postal_value_scores.reset_index()
postal_value_scores = postal_value_scores[postal_value_scores['Property_Count'] >= 10]

print(f"Districts with sufficient data: {len(postal_value_scores)}")

# Sort and get top/bottom 3
postal_value_scores_sorted = postal_value_scores.sort_values('Value_Score_Mean', ascending=False)
top_3_districts = postal_value_scores_sorted.head(3)
bottom_3_districts = postal_value_scores_sorted.tail(3)

print(f"\n🏆 TOP 3 BEST VALUE DISTRICTS:")
print("=" * 50)
print("🎯 Based on model predictions WITHOUT postal district features")
print("📊 Grouped by postal district AFTER prediction")
for _, row in top_3_districts.iterrows():
    percentage_diff = ((row['Value_Score_Mean'] - 100) / 100 * 100)
    print(f"\nDistrict {int(row['Postal_District_Cat'])}:")
    print(f"  Avg Value Score: {row['Value_Score_Mean']:.1f} (from {row['Property_Count']} properties)")
    print(f"  Rents are {percentage_diff:+.1f}% {'lower' if percentage_diff > 0 else 'higher'} than predicted")
    print(f"  Actual Avg Rent: ${row['Actual_Rent_Mean']:,.0f}")
    print(f"  Predicted Avg Rent: ${row['Predicted_Rent_Mean']:,.0f}")

print(f"\n🔻 BOTTOM 3 WORST VALUE DISTRICTS:")
print("=" * 50)
print("🎯 Based on model predictions WITHOUT postal district features")
print("📊 Grouped by postal district AFTER prediction")
for _, row in bottom_3_districts.iterrows():
    percentage_diff = ((row['Value_Score_Mean'] - 100) / 100 * 100)
    print(f"\nDistrict {int(row['Postal_District_Cat'])}:")
    print(f"  Avg Value Score: {row['Value_Score_Mean']:.1f} (from {row['Property_Count']} properties)")
    print(f"  Rents are {percentage_diff:+.1f}% {'lower' if percentage_diff > 0 else 'higher'} than predicted")
    print(f"  Actual Avg Rent: ${row['Actual_Rent_Mean']:,.0f}")
    print(f"  Predicted Avg Rent: ${row['Predicted_Rent_Mean']:,.0f}")

# Create visualization
plt.figure(figsize=(14, 10))
highlight_districts = pd.concat([top_3_districts, bottom_3_districts])

colors = []
for district in highlight_districts['Postal_District_Cat']:
    if district in top_3_districts['Postal_District_Cat'].values:
        colors.append('#2E8B57')  # Green
    else:
        colors.append('#DC143C')  # Red

bars = plt.bar(range(len(highlight_districts)), 
               highlight_districts['Value_Score_Mean'], 
               color=colors, alpha=0.8, edgecolor='black')

plt.xlabel('Postal District', fontsize=13, fontweight='bold')
plt.ylabel('Average Value Score × 100', fontsize=13, fontweight='bold')
plt.title('Top 3 vs Bottom 3 Postal Districts by Value Score\n(Model: 10 Distances + Floor Area + 2 Temporal + Property Type\nGrouping: Postal District (Not in Model))', 
          fontsize=14, fontweight='bold', pad=20)

plt.xticks(range(len(highlight_districts)), 
           [f"District {int(d)}" for d in highlight_districts['Postal_District_Cat']],
           rotation=45, fontsize=11)

for i, (bar, score, count) in enumerate(zip(bars, highlight_districts['Value_Score_Mean'], highlight_districts['Property_Count'])):
    percentage_diff = ((score - 100) / 100 * 100)
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'{score:.1f}\n({percentage_diff:+.1f}%)', 
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.axhline(y=100, color='black', linestyle='--', alpha=0.7, linewidth=2)
plt.text(len(highlight_districts)-0.5, 102, 'Fair Value (100)', 
         ha='right', va='bottom', fontsize=10, fontweight='bold')

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('postal_district_value_scores_correct.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\n🎯 KEY INSIGHT:")
print("=" * 50)
print("The model predicts rent based ONLY on:")
print("  • Amenity distances (10 features)")
print("  • Floor area")
print("  • Temporal features (property age, market time)")
print("  • Property type")
print("\nPostal districts reveal which areas have:")
print("  🟢 HIGHER scores: Rents lower than predicted (good value)")
print("  🔴 LOWER scores: Rents higher than predicted (premium pricing)")

print(f"\n✅ ANALYSIS COMPLETED CORRECTLY!")
print(f"💾 Chart saved as 'postal_district_value_scores_correct.png'")


In [ ]:
# Create visualization
plt.figure(figsize=(14, 10))
highlight_districts = pd.concat([top_3_districts, bottom_3_districts])

colors = []
for district in highlight_districts['Postal_District_Cat']:
    if district in top_3_districts['Postal_District_Cat'].values:
        colors.append('#2E8B57')  # Green
    else:
        colors.append('#DC143C')  # Red

bars = plt.bar(range(len(highlight_districts)), 
               highlight_districts['Value_Score_Mean'], 
               color=colors, alpha=0.8, edgecolor='black')

plt.xlabel('Postal District', fontsize=13, fontweight='bold')
plt.ylabel('Average Value Score × 100', fontsize=13, fontweight='bold')
plt.title('Top 3 vs Bottom 3 Postal Districts by Value Score\n(Model: 10 Distances + Floor Area + 2 Temporal + Property Type\nGrouping: Postal District (Not in Model))', 
          fontsize=14, fontweight='bold', pad=20)

plt.xticks(range(len(highlight_districts)), 
           [f"District {int(d)}" for d in highlight_districts['Postal_District_Cat']],
           rotation=45, fontsize=11)

# REVISED: Larger and more prominent value scores on bars
for i, (bar, score, count) in enumerate(zip(bars, highlight_districts['Value_Score_Mean'], highlight_districts['Property_Count'])):
    percentage_diff = ((score - 100) / 100 * 100)
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'{score:.1f}\n({percentage_diff:+.1f}%)', 
             ha='center', va='bottom', fontsize=12, fontweight='bold',  # Increased fontsize from 10 to 12
             bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8, edgecolor='gray'))  # Added background box

plt.axhline(y=100, color='black', linestyle='--', alpha=0.7, linewidth=2)
plt.text(len(highlight_districts)-0.5, 102, 'Fair Value (100)', 
         ha='right', va='bottom', fontsize=10, fontweight='bold')

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('postal_district_value_scores_correct.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
import folium
from folium.plugins import MarkerCluster
import matplotlib.colors as mcolors
import numpy as np

print("🗺️ CREATING GEOCODED VALUE SCORES MAP")
print("=" * 50)

# Create a base map centered on Singapore
singapore_center = [1.3521, 103.8198]
m = folium.Map(location=singapore_center, zoom_start=11, tiles='OpenStreetMap')

print(f"📊 Total properties to plot: {len(test_df)}")
print(f"🎨 Color coding: Green (>100) vs Red (<100)")

# Create color gradient function
def get_color(value):
    """Return color based on value score"""
    if value >= 100:
        # Green gradient: lighter to darker green
        intensity = min(1.0, (value - 100) / 50)  # Scale from 100-150
        return mcolors.to_hex((0, 0.5 + intensity * 0.5, 0))  # Dark green gradient
    else:
        # Red gradient: lighter to darker red
        intensity = min(1.0, (100 - value) / 50)  # Scale from 50-100
        return mcolors.to_hex((0.8 + intensity * 0.2, 0, 0))  # Dark red gradient

# Add points to the map
for idx, row in test_df.iterrows():
    value_score = row['Value_Score']
    color = get_color(value_score)
    
    # Create popup information
    popup_text = f"""
    <b>Value Score: {value_score:.1f}</b><br>
    District: {int(row['Postal_District_Cat'])}<br>
    Actual Rent: ${row['Monthly Rent ($)']:,.0f}<br>
    Predicted Rent: ${row['Predicted_Rent']:,.0f}<br>
    Floor Area: {row['Floor_Area_SQM']:.0f} SQM
    """
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        popup=folium.Popup(popup_text, max_width=300),
        color=color,
        fillColor=color,
        fillOpacity=0.7,
        weight=1
    ).add_to(m)

# Add legend
legend_html = '''
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 250px; height: 150px; 
     background-color: white; border:2px solid grey; z-index:9999; 
     font-size:14px; padding: 10px">
     
     <p><strong>Value Score Legend</strong></p>
     <p><span style="color: #006400">●</span> >100 (Good Value)</p>
     <p><span style="color: #228B22">●</span> ~125 (Better Value)</p>
     <p><span style="color: #32CD32">●</span> ~150 (Best Value)</p>
     <p><span style="color: #FF6B6B">●</span> ~75 (Lower Value)</p>
     <p><span style="color: #DC143C">●</span> ~50 (Lowest Value)</p>
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))

# Save the map
m.save('singapore_value_scores_map.html')

print(f"\n✅ MAP CREATED SUCCESSFULLY!")
print(f"💾 Saved as 'singapore_value_scores_map.html'")
print(f"🎯 Color Scheme:")
print(f"   🟢 Green: Scores above 100 (darker = better value)")
print(f"   🔴 Red: Scores below 100 (darker = worse value)")
print(f"📊 Statistics:")
print(f"   Properties with score > 100: {len(test_df[test_df['Value_Score'] > 100])}")
print(f"   Properties with score < 100: {len(test_df[test_df['Value_Score'] < 100])}")
print(f"   Average score: {test_df['Value_Score'].mean():.1f}")

# Optional: Create a clustered version for better performance with many points
print(f"\n🔄 Creating clustered version for better performance...")

m_cluster = folium.Map(location=singapore_center, zoom_start=11, tiles='OpenStreetMap')
marker_cluster = MarkerCluster().add_to(m_cluster)

for idx, row in test_df.iterrows():
    value_score = row['Value_Score']
    color = get_color(value_score)
    
    popup_text = f"""
    <b>Value Score: {value_score:.1f}</b><br>
    District: {int(row['Postal_District_Cat'])}<br>
    Actual Rent: ${row['Monthly Rent ($)']:,.0f}
    """
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        popup=folium.Popup(popup_text, max_width=300),
        color=color,
        fillColor=color,
        fillOpacity=0.7,
        weight=1
    ).add_to(marker_cluster)

# Add same legend to clustered map
m_cluster.get_root().html.add_child(folium.Element(legend_html))
m_cluster.save('singapore_value_scores_clustered.html')

print(f"💾 Clustered version saved as 'singapore_value_scores_clustered.html'")
print(f"\n📱 HOW TO USE:")
print(f"   1. Open the HTML file in your web browser")
print(f"   2. Zoom in/out to explore different areas")
print(f"   3. Click on circles to see property details")
print(f"   4. Use clustered version if you have many points for better performance")


In [ ]:
!pip install folium


In [ ]:
print("📊 DISTRICT-WISE MEAN OF ALL FEATURES")
print("=" * 50)

# Create district-wise mean table for all features
district_means = test_df.groupby('Postal_District_Cat').mean(numeric_only=True).round(2)

# Reset index to make Postal_District_Cat a column
district_means = district_means.reset_index()

# Display the table
print(f"Total districts: {len(district_means)}")
print(f"Total properties: {len(test_df)}")
print(f"\nDistrict-wise Mean Table:")

# Display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(district_means)

# If you want to see just the key columns for better readability:
print(f"\n🔑 KEY FEATURES SUMMARY:")
key_columns = ['Postal_District_Cat', 'Value_Score', 'Monthly Rent ($)', 'Predicted_Rent', 'Floor_Area_SQM']
key_means = district_means[key_columns]
print(key_means)

print(f"\n📈 SUMMARY STATISTICS:")
print(f"Average Value Score across all districts: {district_means['Value_Score'].mean():.2f}")
print(f"District with highest Value Score: {district_means.loc[district_means['Value_Score'].idxmax(), 'Postal_District_Cat']} ({district_means['Value_Score'].max():.2f})")
print(f"District with lowest Value Score: {district_means.loc[district_means['Value_Score'].idxmin(), 'Postal_District_Cat']} ({district_means['Value_Score'].min():.2f})")


In [ ]:
print("📊 DISTRICT-WISE AVERAGE FEATURES TABLE")
print("=" * 60)

# Get all distance columns
distance_features = [col for col in test_df.columns if 'distance_to_nearest' in col]

# Define the columns we need
feature_columns = ['Floor_Area_SQM'] + distance_features + ['Days_Since_Min', 'Months_From_Now']

# Create district-wise mean table
district_features = test_df.groupby('Postal_District_Cat')[feature_columns].mean().round(2)

# Reset index to make Postal_District_Cat a column
district_features = district_features.reset_index()

# Rename columns for better readability
column_rename = {
    'Postal_District_Cat': 'District',
    'Floor_Area_SQM': 'Avg_Floor_Area_SQM',
    'Days_Since_Min': 'Avg_Property_Age_Days',
    'Months_From_Now': 'Avg_Months_From_Now'
}
district_features = district_features.rename(columns=column_rename)

# Display the table
print(f"Total districts: {len(district_features)}")
print(f"Features included:")
print(f"  - Average Floor Area (SQM)")
print(f"  - {len(distance_features)} distance features")
print(f"  - Average Property Age (Days)")
print(f"  - Average Months From Now")

print(f"\n{'='*100}")
print(f"{'District':<10} {'Avg Floor Area':<15} {'Avg Property Age':<18} {'Avg Months From Now':<20}", end="")
for i, dist_feat in enumerate(distance_features[:5]):  # First 5 distance features
    print(f" {dist_feat[:15]:<15}", end="")
print()

print(f"{'':<10} {'(SQM)':<15} {'(Days)':<18} {'(Months)':<20}", end="")
for i, dist_feat in enumerate(distance_features[:5]):
    print(f" {'(km)':<15}", end="")
print()
print(f"{'='*100}")

# Display the data
for _, row in district_features.iterrows():
    print(f"{int(row['District']):<10} {row['Avg_Floor_Area_SQM']:<15} {row['Avg_Property_Age_Days']:<18} {row['Avg_Months_From_Now']:<20}", end="")
    for dist_feat in distance_features[:5]:
        print(f" {row[dist_feat]:<15.2f}", end="")
    print()

print(f"\n📋 COMPLETE TABLE (All {len(distance_features)} distance features):")
print(district_features)

print(f"\n📊 SUMMARY:")
print(f"Average Floor Area across districts: {district_features['Avg_Floor_Area_SQM'].mean():.2f} SQM")
print(f"Average Property Age across districts: {district_features['Avg_Property_Age_Days'].mean():.0f} days")
print(f"Range of Floor Areas: {district_features['Avg_Floor_Area_SQM'].min():.1f} - {district_features['Avg_Floor_Area_SQM'].max():.1f} SQM")


In [ ]:
print("🏆 INVERSE RANKING TABLE FOR ALL DISTRICTS")
print("=" * 60)

# Create a copy of the district_features dataframe for ranking
ranking_df = district_features.copy()

# Get the total number of districts
num_districts = len(ranking_df)

print(f"Total districts for ranking: {num_districts}")
print(f"Ranking system: 1st place = {num_districts} points, {num_districts}th place = 1 point")

# Create inverse ranking for each feature
ranking_columns = ['Avg_Floor_Area_SQM', 'Avg_Property_Age_Days', 'Avg_Months_From_Now'] + distance_features

# Initialize ranking columns
for col in ranking_columns:
    rank_col_name = f"{col}_Rank"
    points_col_name = f"{col}_Points"
    
    # For distance features, lower distance is better (higher rank)
    if 'distance' in col:
        # Lower distance gets higher points
        ranking_df[rank_col_name] = ranking_df[col].rank(ascending=True, method='min')
        ranking_df[points_col_name] = num_districts - ranking_df[rank_col_name] + 1
    else:
        # For other features, higher values are generally better
        # Higher floor area = better, lower property age = better, lower months from now = better
        if col == 'Avg_Floor_Area_SQM':
            # Higher floor area is better
            ranking_df[rank_col_name] = ranking_df[col].rank(ascending=False, method='min')
            ranking_df[points_col_name] = num_districts - ranking_df[rank_col_name] + 1
        elif col in ['Avg_Property_Age_Days', 'Avg_Months_From_Now']:
            # Lower values are better (newer properties, closer to min date)
            ranking_df[rank_col_name] = ranking_df[col].rank(ascending=True, method='min')
            ranking_df[points_col_name] = num_districts - ranking_df[rank_col_name] + 1

# Calculate total points for each district
points_columns = [col for col in ranking_df.columns if '_Points' in col]
ranking_df['Total_Points'] = ranking_df[points_columns].sum(axis=1)

# Sort by total points (highest to lowest)
ranking_df = ranking_df.sort_values('Total_Points', ascending=False)

print(f"\n📊 RANKING SUMMARY:")
print(f"Features ranked: {len(ranking_columns)}")
print(f"  - Floor Area (higher = better)")
print(f"  - Property Age (lower = better)") 
print(f"  - Months From Now (lower = better)")
print(f"  - {len(distance_features)} distance features (lower = better)")

# Display the ranking table
print(f"\n{'='*120}")
print(f"{'District':<8} {'Total Points':<12} {'Floor Area':<12} {'Property Age':<14} {'Months From Now':<16}", end="")
for i, dist_feat in enumerate(distance_features[:4]):  # Show first 4 distance features
    short_name = dist_feat.replace('distance_to_nearest_', '')[:10]
    print(f" {short_name:<12}", end="")
print()

print(f"{'':<8} {'':<12} {'Points':<12} {'Points':<14} {'Points':<16}", end="")
for i in range(4):
    print(f" {'Points':<12}", end="")
print()
print(f"{'='*120}")

for _, row in ranking_df.iterrows():
    print(f"{int(row['District']):<8} {row['Total_Points']:<12.0f} {row['Avg_Floor_Area_SQM_Points']:<12.0f} {row['Avg_Property_Age_Days_Points']:<14.0f} {row['Avg_Months_From_Now_Points']:<16.0f}", end="")
    for i, dist_feat in enumerate(distance_features[:4]):
        points_col = f"{dist_feat}_Points"
        print(f" {row[points_col]:<12.0f}", end="")
    print()

# Show top 5 and bottom 5 districts
print(f"\n🏆 TOP 5 DISTRICTS BY TOTAL POINTS:")
top_5 = ranking_df.head(5)[['District', 'Total_Points'] + points_columns]
for _, row in top_5.iterrows():
    print(f"District {int(row['District'])}: {row['Total_Points']:.0f} total points")

print(f"\n🔻 BOTTOM 5 DISTRICTS BY TOTAL POINTS:")
bottom_5 = ranking_df.tail(5)[['District', 'Total_Points'] + points_columns]
for _, row in bottom_5.iterrows():
    print(f"District {int(row['District'])}: {row['Total_Points']:.0f} total points")

# Show detailed ranking for a sample district
print(f"\n🔍 SAMPLE DISTRICT BREAKDOWN (District {int(ranking_df.iloc[0]['District'])} - Ranked #1):")
sample_district = ranking_df.iloc[0]
print(f"Total Points: {sample_district['Total_Points']:.0f}")
for col in ranking_columns[:5]:  # Show first 5 features
    points_col = f"{col}_Points"
    rank_col = f"{col}_Rank"
    print(f"  {col}: {sample_district[points_col]:.0f} points (Rank {int(sample_district[rank_col])})")

print(f"\n💾 Complete ranking table saved in 'ranking_df' variable")
print(f"   Contains all {len(points_columns)} point columns and rankings")


In [ ]:
print("🎯 CALCULATING AGE-BASED WEIGHTED SCORES FOR 20-29 RENTALS")
print("=" * 60)

# Define weights for 20-29 age group in rentals (from your tables)
weights_20_29_rental = {
    'Proximity to MRT/Transport': (5, 4.03),      # (Rental, Age_20_29)
    'Budget / Price / Rent': (5, 5),              # (Rental, Age_20_29)
    'Proximity to Parents': (4, 5),               # (Rental, Age_20_29)
    'Remaining Lease / Age of Property': (1, 2),  # (Rental, Age_20_29)
    'Floor Level & View': (3.5, 2),               # (Rental, Age_20_29)
    'Size & Layout': (3.5, 3),                    # (Rental, Age_20_29)
    'Condo Facilities': (3.5, 3),                 # (Rental, Age_20_29)
    'Proximity to Amenities': (4.5, 5),           # (Rental, Age_20_29)
    'Proximity to Good Schools': (3, 3),          # (Rental, Age_20_29)
    'Future Development Potential': (1, 4),       # (Rental, Age_20_29)
    'Estate / Project Prestige': (3, 2)           # (Rental, Age_20_29)
}

# Calculate combined weights: (Rental_Weight/5) * (Age_Weight/5)
combined_weights = {}
for factor, (rental_w, age_w) in weights_20_29_rental.items():
    combined_weight = (rental_w / 5) * (age_w / 5)
    combined_weights[factor] = combined_weight

print("COMBINED WEIGHTS CALCULATION:")
for factor, weight in combined_weights.items():
    rental_w, age_w = weights_20_29_rental[factor]
    print(f"{factor:<35}: ({rental_w}/5) × ({age_w}/5) = {weight:.3f}")


In [ ]:
print(f"\n🔍 MAPPING AVAILABLE FEATURES FROM ranking_df")
print("=" * 50)

# Map available features to factor categories
feature_mapping = {
    # Transportation features
    'distance_to_nearest_mrt_stations': 'Proximity to MRT/Transport',
    'distance_to_nearest_taxi_stops': 'Proximity to MRT/Transport',
    
    # School features
    'distance_to_nearest_schools': 'Proximity to Good Schools',
    
    # Amenities features
    'distance_to_nearest_parks': 'Proximity to Amenities',
    'distance_to_nearest_gyms': 'Proximity to Amenities',
    'distance_to_nearest_cycling_paths': 'Proximity to Amenities',
    'distance_to_nearest_disability_services': 'Proximity to Amenities',
    'distance_to_nearest_tourist_attractions': 'Proximity to Amenities',
    'distance_to_nearest_hawker_centres': 'Proximity to Amenities',
    'distance_to_nearest_supermarkets': 'Proximity to Amenities',
    
    # Property features
    'Avg_Floor_Area_SQM': 'Size & Layout',
    'Avg_Property_Age_Days': 'Remaining Lease / Age of Property',
}

# Show available mappings with weights
available_features = []
print("AVAILABLE FEATURE MAPPINGS:")
for feature, factor in feature_mapping.items():
    if feature in ranking_df.columns and f"{feature}_Points" in ranking_df.columns:
        weight = combined_weights.get(factor, 0)
        available_features.append(feature)
        print(f"  ✅ {feature:<45} → {factor:<35} (weight: {weight:.3f})")

# Identify unavailable factors
unmapped_factors = set(combined_weights.keys()) - set(feature_mapping.values())
if unmapped_factors:
    print(f"\n⚠️  UNAVAILABLE FACTORS (no corresponding features in data):")
    for factor in unmapped_factors:
        print(f"  ❌ {factor} (weight: {combined_weights[factor]:.3f})")

print(f"\n📊 SUMMARY:")
print(f"Total factors defined: {len(combined_weights)}")
print(f"Factors mapped to available features: {len(set(feature_mapping.values()) & set(combined_weights.keys()))}")
print(f"Available features with points: {len(available_features)}")


In [ ]:
print(f"\n🧮 CALCULATING WEIGHTED NUMERATORS")
print("=" * 50)

# Calculate weighted numerator for each district
district_calculations = []

for district in ranking_df['District'].unique():
    district_data = ranking_df[ranking_df['District'] == district].iloc[0]
    weighted_numerator = 0
    feature_contributions = []
    
    for feature, factor in feature_mapping.items():
        if feature in ranking_df.columns and f"{feature}_Points" in ranking_df.columns:
            points = district_data[f"{feature}_Points"]
            weight = combined_weights.get(factor, 0)
            feature_contribution = points * weight
            weighted_numerator += feature_contribution
            feature_contributions.append({
                'feature': feature,
                'factor': factor,
                'points': points,
                'weight': weight,
                'contribution': feature_contribution
            })
    
    district_calculations.append({
        'District': district,
        'Weighted_Numerator': weighted_numerator,
        'Features_Used': len(feature_contributions),
        'Feature_Contributions': feature_contributions
    })

weighted_df = pd.DataFrame(district_calculations)

print(f"Weighted numerators calculated for {len(weighted_df)} districts")
print(f"Average weighted numerator: {weighted_df['Weighted_Numerator'].mean():.2f}")


In [ ]:
print(f"\n🧮 STEP-BY-STEP CALCULATION FOR SAMPLE DISTRICT")
print("=" * 60)

# Let's take the first district as an example
sample_district = ranking_df['District'].iloc[0]
district_data = ranking_df[ranking_df['District'] == sample_district].iloc[0]

print(f"Calculating for District {int(sample_district)}:")
print(f"{'Feature':<50} {'Factor':<35} {'Points':<8} {'Weight':<8} {'Contribution':<12}")
print("-" * 120)

weighted_numerator = 0
feature_contributions = []

for feature, factor in feature_mapping.items():
    if feature in ranking_df.columns and f"{feature}_Points" in ranking_df.columns:
        points = district_data[f"{feature}_Points"]
        weight = combined_weights.get(factor, 0)
        feature_contribution = points * weight
        weighted_numerator += feature_contribution
        
        print(f"{feature:<50} {factor:<35} {points:<8} {weight:<8.3f} {feature_contribution:<12.2f}")
        
        feature_contributions.append({
            'feature': feature,
            'factor': factor,
            'points': points,
            'weight': weight,
            'contribution': feature_contribution
        })

print("-" * 120)
print(f"{'TOTAL WEIGHTED NUMERATOR:':<93} {weighted_numerator:.2f}")
print(f"{'NUMBER OF FEATURES USED:':<93} {len(feature_contributions)}")

# Now let's show the actual values for this district
print(f"\n📊 ACTUAL VALUES FOR DISTRICT {int(sample_district)}:")
print(f"{'Feature':<50} {'Actual Value':<15} {'Rank':<8} {'Points':<8}")
print("-" * 90)

for feature in feature_mapping.keys():
    if feature in ranking_df.columns:
        actual_value = district_data[feature]
        rank = district_data.get(f"{feature}_Rank", "N/A")
        points = district_data.get(f"{feature}_Points", "N/A")
        print(f"{feature:<50} {actual_value:<15.2f} {rank:<8} {points:<8}")

# Show by factor category totals
print(f"\n🔍 BREAKDOWN BY FACTOR CATEGORY:")
print(f"{'Factor Category':<35} {'Total Contribution':<15} {'Features Count':<15}")
print("-" * 65)

factor_totals = {}
for contrib in feature_contributions:
    factor = contrib['factor']
    if factor not in factor_totals:
        factor_totals[factor] = {'total': 0, 'count': 0}
    factor_totals[factor]['total'] += contrib['contribution']
    factor_totals[factor]['count'] += 1

for factor, data in factor_totals.items():
    print(f"{factor:<35} {data['total']:<15.2f} {data['count']:<15}")

print(f"\n📈 VERIFICATION CALCULATION:")
print(f"Sum of all feature contributions: {sum(contrib['contribution'] for contrib in feature_contributions):.2f}")
print(f"Final weighted numerator: {weighted_numerator:.2f}")
print(f"✓ These should match: {abs(sum(contrib['contribution'] for contrib in feature_contributions) - weighted_numerator) < 0.01}")

# Show the combined weights used
print(f"\n🎯 COMBINED WEIGHTS USED IN CALCULATION:")
for factor, weight in combined_weights.items():
    features_for_factor = [f for f, fac in feature_mapping.items() if fac == factor and f in ranking_df.columns and f"{f}_Points" in ranking_df.columns]
    if features_for_factor:
        print(f"{factor:<35} {weight:.3f} (used by {len(features_for_factor)} features)")


In [ ]:
print(f"\n💰 COMBINING WITH ACTUAL PRICES")
print("=" * 50)

# Get average actual rent for each district from test_df
district_prices = test_df.groupby('Postal_District_Cat')['Monthly Rent ($)'].mean().reset_index()
district_prices.columns = ['District', 'Avg_Actual_Rent']

# Merge with weighted numerators
final_scores_df = weighted_df.merge(district_prices, on='District', how='left')

# Calculate final score: Weighted_Numerator / Avg_Actual_Rent
final_scores_df['Final_Value_Score'] = final_scores_df['Weighted_Numerator'] / final_scores_df['Avg_Actual_Rent']

# Sort by final score (higher = better value)
final_scores_df = final_scores_df.sort_values('Final_Value_Score', ascending=False)

print(f"📊 FINAL VALUE SCORES FOR 20-29 RENTAL PREFERENCES")
print("=" * 80)
print(f"{'District':<8} {'Weighted Num':<14} {'Avg Rent':<12} {'Final Score':<15} {'Features':<10} {'Rank':<6}")
print("-" * 80)

for rank, (idx, row) in enumerate(final_scores_df.iterrows(), 1):
    print(f"{int(row['District']):<8} {row['Weighted_Numerator']:<14.1f} ${row['Avg_Actual_Rent']:<11.0f} {row['Final_Value_Score']:<15.6f} {row['Features_Used']:<10} #{rank:<5}")


In [ ]:
print(f"\n🔍 DETAILED BREAKDOWN FOR TOP DISTRICT")
print("=" * 60)
top_district = final_scores_df.iloc[0]
print(f"District {int(top_district['District'])} - Rank #1")
print(f"Final Score: {top_district['Final_Value_Score']:.6f}")
print(f"Weighted Numerator: {top_district['Weighted_Numerator']:.1f}")
print(f"Average Rent: ${top_district['Avg_Actual_Rent']:.0f}")
print(f"Features Used: {top_district['Features_Used']}")

print(f"\nFeature Contributions:")
contributions = top_district['Feature_Contributions']
for contrib in sorted(contributions, key=lambda x: x['contribution'], reverse=True):
    print(f"  {contrib['feature']:<45} → {contrib['contribution']:<6.1f} ({contrib['points']} pts × {contrib['weight']:.3f} wt)")

print(f"\n🏆 TOP 5 DISTRICTS FOR 20-29 RENTAL VALUE:")
top_5 = final_scores_df.head(5)
for _, row in top_5.iterrows():
    score_per_k = (row['Final_Value_Score'] * 1000)  # Score per $1000 of rent
    print(f"District {int(row['District'])}: {row['Final_Value_Score']:.6f} (${score_per_k:.2f} value per $1k rent)")

print(f"\n📈 SUMMARY STATISTICS")
print("=" * 50)
print(f"Total districts analyzed: {len(final_scores_df)}")
print(f"Average weighted numerator: {final_scores_df['Weighted_Numerator'].mean():.1f}")
print(f"Average actual rent: ${final_scores_df['Avg_Actual_Rent'].mean():.0f}")
print(f"Average final score: {final_scores_df['Final_Value_Score'].mean():.6f}")
print(f"Score range: {final_scores_df['Final_Value_Score'].min():.6f} - {final_scores_df['Final_Value_Score'].max():.6f}")


In [ ]:
print("🎯 CALCULATING AGE-BASED WEIGHTED SCORES FOR 40-49 RENTALS")
print("=" * 60)

# Define weights for 40-49 age group in rentals (from your tables)
weights_40_49_rental = {
    'Proximity to MRT/Transport': (5, 2.58),      # (Rental, Age_40_49)
    'Budget / Price / Rent': (5, 3),              # (Rental, Age_40_49)
    'Proximity to Parents': (4, 2),               # (Rental, Age_40_49)
    'Remaining Lease / Age of Property': (1, 4),  # (Rental, Age_40_49)
    'Floor Level & View': (3.5, 4),               # (Rental, Age_40_49)
    'Size & Layout': (3.5, 5),                    # (Rental, Age_40_49)
    'Condo Facilities': (3.5, 4),                 # (Rental, Age_40_49)
    'Proximity to Amenities': (4.5, 5),           # (Rental, Age_40_49)
    'Proximity to Good Schools': (3, 5),          # (Rental, Age_40_49)
    'Future Development Potential': (1, 2),       # (Rental, Age_40_49)
    'Estate / Project Prestige': (3, 4)           # (Rental, Age_40_49)
}

# Calculate combined weights: (Rental_Weight/5) * (Age_Weight/5)
combined_weights_40_49 = {}
for factor, (rental_w, age_w) in weights_40_49_rental.items():
    combined_weight = (rental_w / 5) * (age_w / 5)
    combined_weights_40_49[factor] = combined_weight

print("COMBINED WEIGHTS FOR 40-49 AGE GROUP:")
for factor, weight in combined_weights_40_49.items():
    rental_w, age_w = weights_40_49_rental[factor]
    print(f"{factor:<35}: ({rental_w}/5) × ({age_w}/5) = {weight:.3f}")

print(f"\n📊 COMPARISON OF WEIGHTS:")
print(f"{'Factor':<35} {'20-29 Weight':<12} {'40-49 Weight':<12} {'Difference':<12}")
print("-" * 75)
for factor in combined_weights.keys():
    w_20_29 = combined_weights[factor]
    w_40_49 = combined_weights_40_49.get(factor, 0)
    diff = w_40_49 - w_20_29
    print(f"{factor:<35} {w_20_29:<12.3f} {w_40_49:<12.3f} {diff:+.3f}")


In [ ]:
print(f"\n🧮 STEP-BY-STEP CALCULATION FOR 40-49 COHORT - SAMPLE DISTRICT")
print("=" * 70)

# Use the same sample district for comparison
sample_district = ranking_df['District'].iloc[0]
district_data = ranking_df[ranking_df['District'] == sample_district].iloc[0]

print(f"Calculating for District {int(sample_district)} - 40-49 Age Group:")
print(f"{'Feature':<50} {'Factor':<35} {'Points':<8} {'Weight':<8} {'Contribution':<12}")
print("-" * 125)

weighted_numerator_40_49 = 0
feature_contributions_40_49 = []

for feature, factor in feature_mapping.items():
    if feature in ranking_df.columns and f"{feature}_Points" in ranking_df.columns:
        points = district_data[f"{feature}_Points"]
        weight = combined_weights_40_49.get(factor, 0)
        feature_contribution = points * weight
        weighted_numerator_40_49 += feature_contribution
        
        print(f"{feature:<50} {factor:<35} {points:<8} {weight:<8.3f} {feature_contribution:<12.2f}")
        
        feature_contributions_40_49.append({
            'feature': feature,
            'factor': factor,
            'points': points,
            'weight': weight,
            'contribution': feature_contribution
        })

print("-" * 125)
print(f"{'TOTAL WEIGHTED NUMERATOR (40-49):':<93} {weighted_numerator_40_49:.2f}")
print(f"{'NUMBER OF FEATURES USED:':<93} {len(feature_contributions_40_49)}")


In [ ]:
print(f"\n🔍 COMPARISON OF BOTH AGE GROUPS FOR DISTRICT {int(sample_district)}")
print("=" * 80)
print(f"{'Feature':<50} {'20-29 Contrib':<12} {'40-49 Contrib':<12} {'Difference':<12}")
print("-" * 90)

# Get the 20-29 contributions for the same district
weighted_numerator_20_29 = 0
for contrib in feature_contributions:
    weighted_numerator_20_29 += contrib['contribution']

# Create comparison
comparison_data = []
for contrib_40_49 in feature_contributions_40_49:
    feature = contrib_40_49['feature']
    contrib_20_29 = next((c['contribution'] for c in feature_contributions if c['feature'] == feature), 0)
    diff = contrib_40_49['contribution'] - contrib_20_29
    comparison_data.append({
        'feature': feature,
        'contrib_20_29': contrib_20_29,
        'contrib_40_49': contrib_40_49['contribution'],
        'diff': diff
    })
    print(f"{feature:<50} {contrib_20_29:<12.2f} {contrib_40_49['contribution']:<12.2f} {diff:+.2f}")

print("-" * 90)
print(f"{'TOTAL:':<50} {weighted_numerator_20_29:<12.2f} {weighted_numerator_40_49:<12.2f} {weighted_numerator_40_49 - weighted_numerator_20_29:+.2f}")


In [ ]:
print(f"\n💰 CALCULATING FINAL SCORES FOR 40-49 COHORT")
print("=" * 50)

# Calculate weighted numerator for all districts for 40-49 cohort
district_calculations_40_49 = []

for district in ranking_df['District'].unique():
    district_data = ranking_df[ranking_df['District'] == district].iloc[0]
    weighted_numerator = 0
    feature_contributions = []
    
    for feature, factor in feature_mapping.items():
        if feature in ranking_df.columns and f"{feature}_Points" in ranking_df.columns:
            points = district_data[f"{feature}_Points"]
            weight = combined_weights_40_49.get(factor, 0)
            feature_contribution = points * weight
            weighted_numerator += feature_contribution
            feature_contributions.append({
                'feature': feature,
                'factor': factor,
                'points': points,
                'weight': weight,
                'contribution': feature_contribution
            })
    
    district_calculations_40_49.append({
        'District': district,
        'Weighted_Numerator_40_49': weighted_numerator,
        'Features_Used': len(feature_contributions),
        'Feature_Contributions': feature_contributions
    })

weighted_df_40_49 = pd.DataFrame(district_calculations_40_49)

# Merge with actual prices and calculate final scores
final_scores_df_40_49 = weighted_df_40_49.merge(district_prices, on='District', how='left')
final_scores_df_40_49['Final_Value_Score_40_49'] = final_scores_df_40_49['Weighted_Numerator_40_49'] / final_scores_df_40_49['Avg_Actual_Rent']
final_scores_df_40_49 = final_scores_df_40_49.sort_values('Final_Value_Score_40_49', ascending=False)

print(f"Weighted numerators calculated for {len(weighted_df_40_49)} districts")
print(f"Average weighted numerator (40-49): {weighted_df_40_49['Weighted_Numerator_40_49'].mean():.2f}")


In [ ]:
print(f"\n🏆 COMPARISON OF BOTH AGE COHORTS - FINAL RANKINGS")
print("=" * 90)

# Merge both results
comparison_df = final_scores_df[['District', 'Weighted_Numerator', 'Final_Value_Score']].copy()
comparison_df = comparison_df.merge(final_scores_df_40_49[['District', 'Weighted_Numerator_40_49', 'Final_Value_Score_40_49']], on='District')
comparison_df = comparison_df.merge(district_prices, on='District')

# Add rankings
comparison_df['Rank_20_29'] = comparison_df['Final_Value_Score'].rank(ascending=False)
comparison_df['Rank_40_49'] = comparison_df['Final_Value_Score_40_49'].rank(ascending=False)
comparison_df['Rank_Difference'] = comparison_df['Rank_40_49'] - comparison_df['Rank_20_29']

print(f"{'District':<8} {'Avg Rent':<10} {'20-29 Score':<12} {'40-49 Score':<12} {'20-29 Rank':<12} {'40-49 Rank':<12} {'Rank Diff':<12}")
print("-" * 90)

for _, row in comparison_df.iterrows():
    rank_diff_str = f"{row['Rank_Difference']:+.0f}"
    print(f"{int(row['District']):<8} ${row['Avg_Actual_Rent']:<9.0f} {row['Final_Value_Score']:<12.6f} {row['Final_Value_Score_40_49']:<12.6f} #{row['Rank_20_29']:<11.0f} #{row['Rank_40_49']:<11.0f} {rank_diff_str:<12}")

print(f"\n📈 KEY DIFFERENCES IN PREFERENCES:")
print(f"Factors that increased for 40-49:")
print(f"  • Size & Layout: {combined_weights_40_49['Size & Layout']:.3f} vs {combined_weights['Size & Layout']:.3f} (+{(combined_weights_40_49['Size & Layout'] - combined_weights['Size & Layout']):+.3f})")
print(f"  • Good Schools: {combined_weights_40_49['Proximity to Good Schools']:.3f} vs {combined_weights['Proximity to Good Schools']:.3f} (+{(combined_weights_40_49['Proximity to Good Schools'] - combined_weights['Proximity to Good Schools']):+.3f})")
print(f"  • Property Age: {combined_weights_40_49['Remaining Lease / Age of Property']:.3f} vs {combined_weights['Remaining Lease / Age of Property']:.3f} (+{(combined_weights_40_49['Remaining Lease / Age of Property'] - combined_weights['Remaining Lease / Age of Property']):+.3f})")

print(f"\nFactors that decreased for 40-49:")
print(f"  • Transport: {combined_weights_40_49['Proximity to MRT/Transport']:.3f} vs {combined_weights['Proximity to MRT/Transport']:.3f} ({(combined_weights_40_49['Proximity to MRT/Transport'] - combined_weights['Proximity to MRT/Transport']):+.3f})")
print(f"  • Parents: {combined_weights_40_49['Proximity to Parents']:.3f} vs {combined_weights['Proximity to Parents']:.3f} ({(combined_weights_40_49['Proximity to Parents'] - combined_weights['Proximity to Parents']):+.3f})")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("📊 CREATING PRESENTATION-READY VISUALIZATIONS")
print("=" * 50)

# Create comparison dataframe with top 15 districts for clarity
top_n = 15
comparison_ranked = comparison_df.nlargest(top_n, 'Final_Value_Score')[['District', 'Final_Value_Score', 'Final_Value_Score_40_49', 'Avg_Actual_Rent']].copy()
comparison_ranked = comparison_ranked.sort_values('Final_Value_Score', ascending=True)

# Visualization 1: Side-by-Side Bar Chart
plt.figure(figsize=(14, 10))
x_pos = np.arange(len(comparison_ranked))
bar_width = 0.35

bars1 = plt.barh(x_pos - bar_width/2, comparison_ranked['Final_Value_Score'], 
                 bar_width, label='Age 20-29', color='#3498db', alpha=0.8)
bars2 = plt.barh(x_pos + bar_width/2, comparison_ranked['Final_Value_Score_40_49'], 
                 bar_width, label='Age 40-49', color='#e74c3c', alpha=0.8)

plt.xlabel('Value Score (Higher = Better Value)', fontsize=12, fontweight='bold')
plt.ylabel('District', fontsize=12, fontweight='bold')
plt.title('Top Districts by Value Score: 20-29 vs 40-49 Age Groups\n(Rental Preferences)', 
          fontsize=14, fontweight='bold', pad=20)
plt.yticks(x_pos, [f'District {int(d)}' for d in comparison_ranked['District']])
plt.legend()

# Add value labels on bars
for i, (bar1, bar2) in enumerate(zip(bars1, bars2)):
    plt.text(bar1.get_width() + 0.0001, bar1.get_y() + bar1.get_height()/2, 
             f'{bar1.get_width():.4f}', ha='left', va='center', fontsize=9, fontweight='bold')
    plt.text(bar2.get_width() + 0.0001, bar2.get_y() + bar2.get_height()/2, 
             f'{bar2.get_width():.4f}', ha='left', va='center', fontsize=9, fontweight='bold')

plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('age_cohort_comparison.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
# Visualization 2: Rank Change Scatter Plot
plt.figure(figsize=(12, 8))

# Color points based on rank improvement/worsening
colors = []
for _, row in comparison_df.iterrows():
    if row['Rank_Difference'] < 0:  # Improved rank for 40-49
        colors.append('#2ecc71')  # Green
    elif row['Rank_Difference'] > 0:  # Worsened rank for 40-49
        colors.append('#e74c3c')  # Red
    else:
        colors.append('#f39c12')  # Yellow

scatter = plt.scatter(comparison_df['Final_Value_Score'], 
                     comparison_df['Final_Value_Score_40_49'],
                     c=colors, s=100, alpha=0.7, edgecolors='black', linewidth=0.5)

# Add district labels
for i, row in comparison_df.iterrows():
    plt.annotate(f"D{int(row['District'])}", 
                (row['Final_Value_Score'], row['Final_Value_Score_40_49']),
                xytext=(5, 5), textcoords='offset points', fontsize=9, fontweight='bold')

# Add reference line
max_score = max(comparison_df['Final_Value_Score'].max(), comparison_df['Final_Value_Score_40_49'].max())
plt.plot([0, max_score], [0, max_score], 'k--', alpha=0.5, label='Equal Preference')

plt.xlabel('Value Score - Age 20-29', fontsize=12, fontweight='bold')
plt.ylabel('Value Score - Age 40-49', fontsize=12, fontweight='bold')
plt.title('District Value Scores: 20-29 vs 40-49 Preferences\n(Green: Better for 40-49, Red: Better for 20-29)', 
          fontsize=14, fontweight='bold', pad=20)

# Custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71', markersize=10, label='Better for 40-49'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10, label='Better for 20-29'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#f39c12', markersize=10, label='No Change'),
    Line2D([0], [0], color='k', linestyle='--', label='Equal Preference')
]
plt.legend(handles=legend_elements, loc='lower right')

plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('age_preference_scatter.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
# Visualization 3: Radar Chart of Weight Differences
from math import pi

# Prepare data for radar chart
factors = list(combined_weights.keys())
weights_20_29 = [combined_weights[f] for f in factors]
weights_40_49 = [combined_weights_40_49[f] for f in factors]

# Number of variables
categories = factors
N = len(categories)

# What will be the angle of each axis in the plot
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Complete the circle

# Add the first value at the end to close the circle
weights_20_29 += weights_20_29[:1]
weights_40_49 += weights_40_49[:1]

# Initialise the spider plot
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(projection='polar'))

# Plot data
ax.plot(angles, weights_20_29, 'o-', linewidth=2, label='20-29 Age Group', color='#3498db')
ax.fill(angles, weights_20_29, alpha=0.25, color='#3498db')
ax.plot(angles, weights_40_49, 'o-', linewidth=2, label='40-49 Age Group', color='#e74c3c')
ax.fill(angles, weights_40_49, alpha=0.25, color='#e74c3c')

# Add labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_yticklabels([])

# Add title
plt.title('Feature Weight Comparison: 20-29 vs 40-49 Age Groups\n(Rental Preferences)', 
          size=14, fontweight='bold', pad=20)

# Add legend
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
plt.savefig('weight_comparison_radar.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
# Visualization 4: Clean Table for Top Districts
print(f"\n🏆 TOP DISTRICTS FOR EACH AGE GROUP - PRESENTATION TABLE")
print("=" * 80)

# Get top 10 for each age group
top_20_29 = comparison_df.nlargest(10, 'Final_Value_Score')[['District', 'Final_Value_Score', 'Avg_Actual_Rent']]
top_40_49 = comparison_df.nlargest(10, 'Final_Value_Score_40_49')[['District', 'Final_Value_Score_40_49', 'Avg_Actual_Rent']]

print(f"\n📊 TOP 10 DISTRICTS FOR 20-29 AGE GROUP:")
print(f"{'Rank':<6} {'District':<10} {'Value Score':<15} {'Avg Rent':<12}")
print("-" * 50)
for i, (idx, row) in enumerate(top_20_29.iterrows(), 1):
    print(f"{i:<6} {int(row['District']):<10} {row['Final_Value_Score']:<15.6f} ${row['Avg_Actual_Rent']:<11.0f}")

print(f"\n📊 TOP 10 DISTRICTS FOR 40-49 AGE GROUP:")
print(f"{'Rank':<6} {'District':<10} {'Value Score':<15} {'Avg Rent':<12}")
print("-" * 50)
for i, (idx, row) in enumerate(top_40_49.iterrows(), 1):
    print(f"{i:<6} {int(row['District']):<10} {row['Final_Value_Score_40_49']:<15.6f} ${row['Avg_Actual_Rent']:<11.0f}")

print(f"\n🔄 DISTRICTS WITH BIGGEST RANK CHANGES:")
biggest_changes = comparison_df.nlargest(5, 'Rank_Difference')[['District', 'Rank_20_29', 'Rank_40_49', 'Rank_Difference']]
print(f"{'District':<10} {'20-29 Rank':<12} {'40-49 Rank':<12} {'Change':<12}")
print("-" * 50)
for _, row in biggest_changes.iterrows():
    print(f"{int(row['District']):<10} #{row['Rank_20_29']:<11.0f} #{row['Rank_40_49']:<11.0f} {row['Rank_Difference']:+.0f} positions")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("📊 CREATING FOCUSED COMPARISON VISUALIZATION")
print("=" * 50)

# Get top 4 and bottom 4 districts for each age group
top_4_20_29 = comparison_df.nlargest(4, 'Final_Value_Score')
bottom_4_20_29 = comparison_df.nsmallest(4, 'Final_Value_Score')

# Combine and remove duplicates
focus_districts = pd.concat([top_4_20_29, bottom_4_20_29]).drop_duplicates()
focus_districts = focus_districts.sort_values('Final_Value_Score', ascending=True)

print(f"Focusing on {len(focus_districts)} districts: Top 4 and Bottom 4")

# Visualization: Side-by-Side Bar Chart
plt.figure(figsize=(12, 8))
x_pos = np.arange(len(focus_districts))
bar_width = 0.4

bars1 = plt.barh(x_pos - bar_width/2, focus_districts['Final_Value_Score'], 
                 bar_width, label='Age 20-29', color='#3498db', alpha=0.8, edgecolor='black')
bars2 = plt.barh(x_pos + bar_width/2, focus_districts['Final_Value_Score_40_49'], 
                 bar_width, label='Age 40-49', color='#e74c3c', alpha=0.8, edgecolor='black')

plt.xlabel('Value Score (Higher = Better Value)', fontsize=14, fontweight='bold')
plt.ylabel('District', fontsize=14, fontweight='bold')
plt.title('Value Score Comparison: Top 4 vs Bottom 4 Districts\n20-29 vs 40-49 Age Groups', 
          fontsize=16, fontweight='bold', pad=25)
plt.yticks(x_pos, [f'District {int(d)}' for d in focus_districts['District']], fontsize=12)
plt.legend(fontsize=12, loc='lower right')

# Add LARGE value labels on bars with better positioning
for i, (bar1, bar2) in enumerate(zip(bars1, bars2)):
    # Position labels inside the bars for better readability
    x_pos_1 = bar1.get_width() * 0.5  # Middle of the bar
    x_pos_2 = bar2.get_width() * 0.5  # Middle of the bar
    
    plt.text(x_pos_1, bar1.get_y() + bar1.get_height()/2, 
             f'{bar1.get_width():.4f}', 
             ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    plt.text(x_pos_2, bar2.get_y() + bar2.get_height()/2, 
             f'{bar2.get_width():.4f}', 
             ha='center', va='center', fontsize=11, fontweight='bold', color='white')

# Add average rent information as text
plt.figtext(0.02, 0.02, 
           f"Top districts average rent: ${focus_districts.nlargest(4, 'Final_Value_Score')['Avg_Actual_Rent'].mean():.0f}\n"
           f"Bottom districts average rent: ${focus_districts.nsmallest(4, 'Final_Value_Score')['Avg_Actual_Rent'].mean():.0f}",
           fontsize=10, style='italic', bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray"))

plt.grid(axis='x', alpha=0.3)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])  # Make room for the footer text
plt.savefig('focused_age_cohort_comparison.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Print the data for verification
print(f"\n📊 FOCUS DISTRICTS DATA:")
print(f"{'District':<10} {'20-29 Score':<12} {'40-49 Score':<12} {'Avg Rent':<12}")
print("-" * 50)
for _, row in focus_districts.iterrows():
    print(f"{int(row['District']):<10} {row['Final_Value_Score']:<12.6f} {row['Final_Value_Score_40_49']:<12.6f} ${row['Avg_Actual_Rent']:<11.0f}")

print(f"\n🎯 KEY OBSERVATIONS:")
top_districts = focus_districts.nlargest(4, 'Final_Value_Score')
bottom_districts = focus_districts.nsmallest(4, 'Final_Value_Score')

print(f"Top 4 Districts (Best Value):")
for _, row in top_districts.iterrows():
    diff = row['Final_Value_Score_40_49'] - row['Final_Value_Score']
    print(f"  District {int(row['District'])}: 20-29={row['Final_Value_Score']:.4f}, 40-49={row['Final_Value_Score_40_49']:.4f} (Δ={diff:+.4f})")

print(f"\nBottom 4 Districts (Worst Value):")
for _, row in bottom_districts.iterrows():
    diff = row['Final_Value_Score_40_49'] - row['Final_Value_Score']
    print(f"  District {int(row['District'])}: 20-29={row['Final_Value_Score']:.4f}, 40-49={row['Final_Value_Score_40_49']:.4f} (Δ={diff:+.4f})")


In [ ]:
# Visualization 2: Rank Change Scatter Plot - PRESENTATION VERSION
plt.figure(figsize=(14, 10))

# Color points based on rank improvement/worsening
colors = []
sizes = []  # Vary sizes for better visibility
for _, row in comparison_df.iterrows():
    if row['Rank_Difference'] < 0:  # Improved rank for 40-49
        colors.append('#27ae60')  # Green - brighter for presentations
        sizes.append(150)  # Larger for important points
    elif row['Rank_Difference'] > 0:  # Worsened rank for 40-49
        colors.append('#e74c3c')  # Red
        sizes.append(150)
    else:
        colors.append('#f39c12')  # Yellow
        sizes.append(120)

scatter = plt.scatter(comparison_df['Final_Value_Score'], 
                     comparison_df['Final_Value_Score_40_49'],
                     c=colors, s=sizes, alpha=0.8, edgecolors='black', linewidth=1.5)

# Add LARGER district labels with better positioning
for i, row in comparison_df.iterrows():
    offset_x = 0.0002  # Smaller offset for tighter zoom
    offset_y = 0.0002
    
    plt.annotate(f"D{int(row['District'])}", 
                (row['Final_Value_Score'], row['Final_Value_Score_40_49']),
                xytext=(offset_x, offset_y), 
                textcoords='offset points', 
                fontsize=11,  # Increased from 9 to 11
                fontweight='bold',
                bbox=dict(boxstyle="round,pad=0.2", facecolor='white', alpha=0.8, edgecolor='none'))

# Add reference line
max_score = max(comparison_df['Final_Value_Score'].max(), comparison_df['Final_Value_Score_40_49'].max())
min_score = min(comparison_df['Final_Value_Score'].min(), comparison_df['Final_Value_Score_40_49'].min())

# Set tighter axis limits to "zoom in" on the data
padding = (max_score - min_score) * 0.05  # 5% padding
plt.xlim(min_score - padding, max_score + padding)
plt.ylim(min_score - padding, max_score + padding)

plt.plot([min_score - padding, max_score + padding], 
         [min_score - padding, max_score + padding], 
         'k--', alpha=0.5, linewidth=2, label='Equal Preference')

# Larger axis labels
plt.xlabel('Value Score - Age 20-29', fontsize=14, fontweight='bold')
plt.ylabel('Value Score - Age 40-49', fontsize=14, fontweight='bold')

# Larger title
plt.title('District Value Scores: 20-29 vs 40-49 Preferences\n(Green: Better for 40-49, Red: Better for 20-29)', 
          fontsize=16, fontweight='bold', pad=25)

# Custom legend with larger text
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#27ae60', markersize=12, label='Better for 40-49'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=12, label='Better for 20-29'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#f39c12', markersize=12, label='No Change'),
    Line2D([0], [0], color='k', linestyle='--', linewidth=2, label='Equal Preference')
]
plt.legend(handles=legend_elements, loc='lower right', fontsize=12, framealpha=0.9)

# Add some key insights as text on the plot
avg_20_29 = comparison_df['Final_Value_Score'].mean()
avg_40_49 = comparison_df['Final_Value_Score_40_49'].mean()

plt.figtext(0.02, 0.02, 
           f"Avg 20-29 Score: {avg_20_29:.4f}\n"
           f"Avg 40-49 Score: {avg_40_49:.4f}\n"
           f"Districts shown: {len(comparison_df)}",
           fontsize=11, style='italic', 
           bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgray", alpha=0.8))

plt.grid(alpha=0.3, linestyle='-', linewidth=0.5)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])  # Make room for footer text
plt.savefig('age_preference_scatter_presentation.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Print key insights
print(f"\n🎯 SCATTER PLOT INSIGHTS:")
above_line = len(comparison_df[comparison_df['Final_Value_Score_40_49'] > comparison_df['Final_Value_Score']])
below_line = len(comparison_df[comparison_df['Final_Value_Score_40_49'] < comparison_df['Final_Value_Score']])
on_line = len(comparison_df) - above_line - below_line

print(f"• {above_line} districts preferred by 40-49 age group")
print(f"• {below_line} districts preferred by 20-29 age group") 
print(f"• {on_line} districts with similar preference")
print(f"• Average score difference: {(avg_40_49 - avg_20_29):+.4f}")
